# D1 · From signed transactions to purchase histories

Use a CPU runtime and run the cells in order. The only input is the original Excel workbook. First reproduce the positive-sale baseline, then build two signed ledgers and classify merchandise credits. Finally replay the credits visible at each cutoff to calculate eligible purchase quantities. The full observed period and the training snapshot have separate output files.

Use the source `ExcelRow` in the evidence workbook to trace a decision. Start with `missingness` and `customer_id_invoices` for unknown customers, `nonC_negative` for stock write-offs, and `P_nonpositive` for prices. The credit checks use `credit_allocations`, `credit_controls`, `credit_control_rows` and `reversal_examples`. [Evidence workbook](../results/transaction_exceptions.xlsx) · [Exception analysis notebook](../docs/technical/transaction_exception_analysis.ipynb) · [Project 2 brief, p. 3](../resource/Group_Project_Tutorial_Recommender_Systems.pdf#page=3)

# D1 · 从带符号交易到购买历史

使用 CPU，按顺序运行全部单元格。唯一输入是原始 Excel 工作簿。先复算正向销售基线，再建立两份带符号账本、分类商品贷项，最后只用截止时点前可见的贷项回放各笔有效购买的件数。全观察期与训练期快照分别导出。

在证据工作簿中可按原表行号 `ExcelRow` 追踪决定。缺客户编号查 `missingness` 和 `customer_id_invoices`，库存核销查 `nonC_negative`，价格问题查 `P_nonpositive`。贷项分配与对照查 `credit_allocations`、`credit_controls`、`credit_control_rows` 和 `reversal_examples`。[证据工作簿](../results/transaction_exceptions.xlsx) · [异常分析 notebook](../docs/technical/transaction_exception_analysis.ipynb) · [项目二要求，第 3 页](../resource/Group_Project_Tutorial_Recommender_Systems.pdf#page=3)

## Locate the original workbook

In Colab, mount Drive and open the project’s `data/raw/Online Retail.xlsx`. When prompted, enter the path from the Files panel. Locally run from the repository root or its `notebooks/` directory. Results go under the chosen project folder.

## 定位原始工作簿

在 Colab 授权挂载 Drive，读取项目文件夹中的 `data/raw/Online Retail.xlsx`。如出现提示，从左侧文件面板复制完整路径。本地可在仓库根目录或 `notebooks/` 中运行。结果写入选定项目文件夹。

In [1]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail")
    workbook = project / "data/raw/Online Retail.xlsx"
    if not workbook.is_file():
        workbook = Path(input("Workbook path / 工作簿完整路径：").strip())
        if not workbook.is_file():
            raise FileNotFoundError("Workbook not found / 找不到工作簿")
        if workbook.parent.name == "raw" and workbook.parent.parent.name == "data":
            project = workbook.parents[2]
        else:
            project = Path(input("Drive project folder for results / 保存结果的 Drive 项目文件夹完整路径：").strip())
            if not project.is_absolute() or not project.is_dir():
                raise ValueError("Select an existing project folder / 请选择已存在的项目文件夹")
else:
    project = Path.cwd()
    if project.name == "notebooks":
        project = project.parent
    workbook = project / "data/raw/Online Retail.xlsx"

if not workbook.is_file():
    raise FileNotFoundError(f"Workbook not found / 找不到工作簿：{workbook}")

output = project / "results/cleaning_audit.csv"
print("Workbook / 工作簿：", workbook)
print("Audit output / 审计表：", output)

Workbook / 工作簿： /tmp/retail-adjusted-nb-Cfm05Qzm/project-replay/data/raw/Online Retail.xlsx
Audit output / 审计表： /tmp/retail-adjusted-nb-Cfm05Qzm/project-replay/results/cleaning_audit.csv


## Verify the source and preserve the row numbers

Check the workbook checksum and eight source fields. Read invoice, product and customer identifiers as text and preserve their exact case. The first data record is Excel row 2. Compare the field counts with `missingness`. A missing description permits a product-coded purchase to survive. A missing country alone also leaves eligibility unchanged.

## 核验输入并保留行号

核对文件指纹与八个原始字段。按文本读取发票、商品与客户编号，保留原始大小写。第一条明细是 Excel 第 2 行。各字段的缺失数可与 `missingness` 对照。有效商品即使缺少描述仍可保留，展示时可用商品编码。仅缺少国家也不影响购买资格。

In [2]:
import hashlib
import pandas as pd
from IPython.display import display

WORKBOOK_SHA256 = "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
RAW_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
]


def read_workbook(path: Path) -> pd.DataFrame:
    with Path(path).open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != WORKBOOK_SHA256:
            raise ValueError("Workbook differs from the source identified in the research plan.")
        stream.seek(0)
        raw = pd.read_excel(stream, sheet_name="Online Retail", engine="openpyxl",
                            dtype={key: "string" for key in ("InvoiceNo", "StockCode", "CustomerID")},
                            keep_default_na=False, na_values=[""])
    if raw.columns.tolist() != RAW_COLUMNS:
        raise ValueError("Unexpected source schema.")
    raw.insert(0, "ExcelRow", range(2, len(raw) + 2))
    return raw
raw = read_workbook(workbook)
display(pd.DataFrame({"type": raw[RAW_COLUMNS].dtypes.astype(str), "missing_rows": raw[RAW_COLUMNS].isna().sum()}))
print(f"{len(raw):,} rows · {raw['InvoiceDate'].min()} — {raw['InvoiceDate'].max()}")

,type,missing_rows
InvoiceNo,string,0
StockCode,string,0
Description,object,1454
Quantity,int64,0
InvoiceDate,datetime64[ns],0
UnitPrice,float64,0
CustomerID,string,135080
Country,object,0


541,909 rows · 2010-12-01 08:26:00 — 2011-12-09 12:50:00


## Reproduce the five-rule baseline

The original audit is a checkpoint. Exactly identical lines are detected using all eight source fields. Raw condition counts can overlap. Only the first matching rule removes a given line. The five-step total leaves 392,692 candidate lines before checking merchandise codes.

## 复算五步基线

用全部八个原始字段识别完全重复行。原表条件可重叠，每条记录只由首次命中的规则移除一次。旧五步审计保留 392,692 条候选交易明细，这是进一步判断商品身份前的校验点。

In [3]:
def purchase_filter_flags(frame: pd.DataFrame) -> dict[str, pd.Series]:
    """Evaluate the five proposed purchase-sample rules in their audit order."""
    return {
        "exact_duplicate": frame.duplicated(subset=RAW_COLUMNS, keep="first"),
        "missing_customer_id": frame["CustomerID"].isna(),
        "cancelled_invoice": frame["InvoiceNo"].astype("string").str.startswith("C", na=False),
        "nonpositive_quantity": frame["Quantity"].le(0).fillna(False),
        "nonpositive_unit_price": frame["UnitPrice"].le(0).fillna(False),
    }


def audit_transactions(frame: pd.DataFrame) -> pd.DataFrame:
    """Count each flag on raw rows and assign each removed row to its first rule."""
    flags = purchase_filter_flags(frame)
    remaining = pd.Series(True, index=frame.index)
    records = []
    for step, (rule, flagged) in enumerate(flags.items(), start=1):
        before = int(remaining.sum())
        removed = int((remaining & flagged).sum())
        remaining &= ~flagged
        records.append({
            "step": step,
            "rule": rule,
            "raw_flagged": int(flagged.sum()),
            "rows_before": before,
            "rows_removed": removed,
            "rows_after": int(remaining.sum()),
        })
    return pd.DataFrame.from_records(records)

baseline = audit_transactions(raw)
display(baseline)

,step,rule,raw_flagged,rows_before,rows_removed,rows_after
0,1,exact_duplicate,5268,541909,5268,536641
1,2,missing_customer_id,135080,536641,135037,401604
2,3,cancelled_invoice,9288,401604,8872,392732
3,4,nonpositive_quantity,10624,392732,0,392732
4,5,nonpositive_unit_price,2517,392732,40,392692


## Read the positive-sale eligibility audit

This first pass establishes which rows could supply a sale before applying credits. `raw_flagged` counts every raw match, `rows_before` counts the rows entering the step, `rows_removed` counts the first exclusions at this step and `rows_after` counts survivors. One row may satisfy several conditions. The final decision table also records the outcome of any later credit match.

## 阅读正向销售资格审计

第一遍筛选确定贷项调整前哪些行可以作为原始销售。`raw_flagged` 统计原表所有命中行；`rows_before` 为进入步骤的行数，`rows_removed` 为该步首次剔除的行数，`rows_after` 为剩余数。一条原始行可能满足多个条件。最后的逐行决定表还会说明匹配贷项后该行的处理结果。

In [4]:
if raw["ExcelRow"].isna().any() or not raw["ExcelRow"].is_unique:
    raise ValueError("ExcelRow must uniquely identify each row of the source worksheet.")
frame = raw.reset_index(drop=True)
flags = purchase_filter_flags(frame)
surviving = pd.Series(True, index=frame.index)
first_rule = pd.Series("retained", index=frame.index, dtype="string")
records = []
def exclude(rule, flag, sheets, reason_en, reason_zh):
    mask = flag.fillna(False)
    removed = mask & surviving
    before = int(surviving.sum())
    count = int(removed.sum())
    first_rule.loc[removed] = rule
    surviving.loc[removed] = False
    records.append({
        "step": len(records) + 1, "rule": rule, "action": "exclude",
        "raw_flagged": int(mask.sum()), "rows_before": before,
        "rows_removed": count, "rows_after": before - count,
        "evidence_sheets": sheets, "reason_en": reason_en, "reason_zh": reason_zh,
    })

def show_steps(entries):
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(pd.DataFrame(entries)[["step", "rule", "raw_flagged", "rows_before", "rows_removed",
                                       "rows_after", "evidence_sheets", "reason_en", "reason_zh"]])

## Duplicates and missing fields

Keep the first exact copy. The 135,080 lines without a customer ID span 3,710 invoices. In `customer_id_invoices`, none of these invoices supplies an ID on another line. Next check each required field separately, including the fields that are complete in this workbook. These checks preserve evidence for the identity, quantity, price and timing of a purchase.

## 重复与缺失：逐字段处理

完全相同的明细保留首条。缺客户编号的 135,080 行涉及 3,710 张发票。`customer_id_invoices` 显示，这些发票的其他行也都没有可补用的编号。随后逐项检查必要字段，原表中缺失数为零的字段也单列，用于核验每笔购买的身份、数量、价格与时间。

In [5]:
exclude("exact_duplicate", flags["exact_duplicate"], "filter_attribution",
        "Keep the first occurrence of an identical eight-field record.",
        "按八个原始字段识别完全相同的明细，保留首次出现的一行。")
exclude("missing_customer_id", flags["missing_customer_id"], "missingness, customer_id_invoices, filter_attribution",
        "An anonymous line cannot be assigned to a customer's history. The source audit retains these lines.",
        "匿名交易无法归入具体客户的历史，原始审计仍涵盖这些行。")
for rule, field, reason_en, reason_zh in [
    ("missing_invoice_no", "InvoiceNo", "An invoice identifier is required to construct a basket.", "缺少发票编号，无法构造购物篮。"),
    ("missing_stock_code", "StockCode", "An item identifier is required to assign a product.", "缺少商品编码，无法确定商品。"),
    ("missing_invoice_date", "InvoiceDate", "The timestamp determines the basket time and eligible history.", "缺少时间，无法确定购物篮发生时间与可用历史。"),
    ("missing_quantity", "Quantity", "The purchased quantity is unknown.", "缺少数量，无法核实购买件数。"),
    ("missing_unit_price", "UnitPrice", "The recorded unit price is unknown.", "缺少单价，无法核实计价金额。"),
]:
    exclude(rule, frame[field].isna(), "missingness", reason_en, reason_zh)
show_steps(records)

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,1,exact_duplicate,5268,541909,5268,536641,filter_attribution,Keep the first occurrence of an identical eight-field record.,按八个原始字段识别完全相同的明细，保留首次出现的一行。
1,2,missing_customer_id,135080,536641,135037,401604,"missingness, customer_id_invoices, filter_attribution",An anonymous line cannot be assigned to a customer's history. The source audit retains these lines.,匿名交易无法归入具体客户的历史，原始审计仍涵盖这些行。
2,3,missing_invoice_no,0,401604,0,401604,missingness,An invoice identifier is required to construct a basket.,缺少发票编号，无法构造购物篮。
3,4,missing_stock_code,0,401604,0,401604,missingness,An item identifier is required to assign a product.,缺少商品编码，无法确定商品。
4,5,missing_invoice_date,0,401604,0,401604,missingness,The timestamp determines the basket time and eligible history.,缺少时间，无法确定购物篮发生时间与可用历史。
5,6,missing_quantity,0,401604,0,401604,missingness,The purchased quantity is unknown.,缺少数量，无法核实购买件数。
6,7,missing_unit_price,0,401604,0,401604,missingness,The recorded unit price is unknown.,缺少单价，无法核实计价金额。


## Identify potential sales, credit notes and stock adjustments

All 9,288 C-prefixed lines have negative quantities. C is the workbook's cancellation-document prefix. Here we exclude those lines from the *positive-sale candidate* set, keeping them available for signed accounting and later matching. Treat the 1,336 non-C negative lines as inventory write-offs: all have zero price and no known customer. Notes include stock counts and damage, while some notes are blank. All three A-prefixed bad-debt lines leave the signed ledgers, including the one with a positive price.

## 区分候选销售、贷项与库存调整

9,288 条 C 前缀明细的数量均为负。C 是原表取消单据的前缀。此处把 C 行从“候选正向销售”中分开，同时保留原记录供后面建账、配对。1,336 条非 C 负数量行按库存核销处理，全部零价且没有客户编号；有的备注写盘点、损坏，也有备注缺失。三个 A 前缀坏账调整行均从账本排除，其中包括单价为正的一行。

In [6]:
start = len(records)
exclude("cancelled_invoice", flags["cancelled_invoice"], "C_invoices, history_summary, case_examples",
        "Route C-prefixed records out of the positive-source stream. Match eligible merchandise credits before determining the purchased quantity.",
        "C 前缀记录不单独产生购买事件。符合条件的商品贷项另行匹配后，再确定最终购买数量。")
exclude("negative_quantity", frame["Quantity"].lt(0), "Q_nonpositive, nonC_negative, note_summary",
        "Negative units do not add purchased items. Notes can describe damage or adjustments without identifying a buyer.",
        "负数量不形成新增购买。备注可能记录损坏或调整，但不足以确认买家。")
exclude("zero_quantity", frame["Quantity"].eq(0), "Q_nonpositive, filter_attribution",
        "Zero units add no purchase event.", "零数量不产生购买事件。")
exclude("negative_unit_price", frame["UnitPrice"].lt(0), "P_nonpositive, case_examples, note_summary",
        "A negative unit price is outside the paid-item definition. Both observed examples record bad-debt adjustments.",
        "负单价不符合计价商品定义，原表两条实例的备注均为坏账调整。")
show_steps(records[start:])

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,8,cancelled_invoice,9288,401604,8872,392732,"C_invoices, history_summary, case_examples",Route C-prefixed records out of the positive-source stream. Match eligible merchandise credits before determining the purchased quantity.,C 前缀记录不单独产生购买事件。符合条件的商品贷项另行匹配后，再确定最终购买数量。
1,9,negative_quantity,10624,392732,0,392732,"Q_nonpositive, nonC_negative, note_summary",Negative units do not add purchased items. Notes can describe damage or adjustments without identifying a buyer.,负数量不形成新增购买。备注可能记录损坏或调整，但不足以确认买家。
2,10,zero_quantity,0,392732,0,392732,"Q_nonpositive, filter_attribution",Zero units add no purchase event.,零数量不产生购买事件。
3,11,negative_unit_price,2,392732,0,392732,"P_nonpositive, case_examples, note_summary",A negative unit price is outside the paid-item definition. Both observed examples record bad-debt adjustments.,负单价不符合计价商品定义，原表两条实例的备注均为坏账调整。


## Zero prices and possible complimentary items

Partition every zero-price line using invoice evidence. The price step reaches 40 identified positive-quantity rows. Six are Manual. Five product lines have the same SKU priced on the invoice, 23 have another priced line, and six product lines have no priced companion. The 23 same-SKU product matches reported in the exception notebook across the whole workbook comprise five identified and 18 anonymous rows. These are two different denominators. The recorded prices do not establish whether an item was a gift. Preserve each qualifying priced line on the invoice.

## 零价与疑似随单赠送

对原表全部零价行检查同票上下文。运行到单价步骤时仍有 40 行，均有客户编号且数量为正。其中 6 行为 Manual；5 条商品明细同票同商品另有正价行，23 条同票另有其他计价行，6 条商品明细同票无正价行。异常分析在全表找出的 23 条「同票同商品还有正价」记录，则是 5 条有编号与 18 条无编号明细之和，两处的 23 采用不同口径。零价原因未注明，只排除零价行，同票中符合条件的计价购买仍保留。

In [7]:
def _zero_price_contexts(raw: pd.DataFrame, cancelled: pd.Series) -> dict[str, pd.Series]:
    """Partition all raw zero-price lines, including lines removed by earlier rules.

    Companions have positive quantity and unit price on a non-C invoice with a
    known invoice number. A same-item match additionally requires the exact
    StockCode and CustomerID, with a missing customer key matching only another
    missing key. A priced line can itself be a fee. Search the complete invoice.
    These comparisons describe a zero price. All four cases have the same
    purchase-label decision, irrespective of where the query cutoff falls.
    """
    zero = raw["UnitPrice"].eq(0).fillna(False)
    manual = raw["StockCode"].isin(["M", "m"])
    keys = ["InvoiceNo", "CustomerID", "StockCode"]
    priced = raw.loc[
        ~cancelled & raw["Quantity"].gt(0) & raw["UnitPrice"].gt(0), keys
    ].dropna(subset=["InvoiceNo"]).drop_duplicates()
    priced_keys = priced.dropna(subset=["StockCode"])
    # Limit the three-key comparison to the small set of zero-price rows.
    same_item = pd.Series(False, index=raw.index)
    same_item.loc[zero] = pd.MultiIndex.from_frame(raw.loc[zero, keys]).isin(
        pd.MultiIndex.from_frame(priced_keys)
    )
    same_invoice = raw["InvoiceNo"].isin(priced["InvoiceNo"])
    return {
        "zero_price_manual": zero & manual,
        "zero_price_same_item": zero & ~manual & same_item,
        "zero_price_other_item": zero & ~manual & ~same_item & same_invoice,
        "zero_price_no_priced_companion": zero & ~manual & ~same_invoice,
    }

start = len(records)
zero_price = _zero_price_contexts(frame, flags["cancelled_invoice"])
zero_sheets = "P_nonpositive, case_examples, history_summary"
exclude("zero_price_manual", zero_price["zero_price_manual"], "P_nonpositive, note_summary",
        "An exact manual-entry code (M or m) carries a recorded zero unit price.",
        "手工记录编码 M 或 m 的记录单价为零。")
exclude("zero_price_same_item", zero_price["zero_price_same_item"], zero_sheets,
        "A priced line shares the invoice, customer key and SKU. The zero-price row adds no paid units.",
        "同票另有计价行匹配该客户键和商品编码，零价行不增加计价购买件数。")
exclude("zero_price_other_item", zero_price["zero_price_other_item"], zero_sheets,
        "This invoice has a priced line, but none matches the customer key and SKU. A priced line can also represent a fee.",
        "同票有计价行，但没有计价行同时匹配本行的客户键和商品编码。计价行也可能是费用。")
exclude("zero_price_no_priced_companion", zero_price["zero_price_no_priced_companion"], zero_sheets,
        "No non-C line with positive quantity and price was found under this exact invoice number.",
        "该发票编号下未查到非 C 且数量和单价均为正的计价行。")
show_steps(records[start:])
assert records[-1]["rows_after"] == baseline.iloc[-1]["rows_after"]

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,12,zero_price_manual,6,392732,6,392726,"P_nonpositive, note_summary",An exact manual-entry code (M or m) carries a recorded zero unit price.,手工记录编码 M 或 m 的记录单价为零。
1,13,zero_price_same_item,23,392726,5,392721,"P_nonpositive, case_examples, history_summary","A priced line shares the invoice, customer key and SKU. The zero-price row adds no paid units.",同票另有计价行匹配该客户键和商品编码，零价行不增加计价购买件数。
2,14,zero_price_other_item,372,392721,23,392698,"P_nonpositive, case_examples, history_summary","This invoice has a priced line, but none matches the customer key and SKU. A priced line can also represent a fee.",同票有计价行，但没有计价行同时匹配本行的客户键和商品编码。计价行也可能是费用。
3,15,zero_price_no_priced_companion,2114,392698,6,392692,"P_nonpositive, case_examples, history_summary",No non-C line with positive quantity and price was found under this exact invoice number.,该发票编号下未查到非 C 且数量和单价均为正的计价行。


## Freeze item-code decisions using training evidence

Review codes against literal descriptions dated from 1 December 2010 through 30 September 2011. The register gives the original Excel row and counts before and after the basic filters in that period. Verify these counts below. Use exact, case-sensitive codes. For example, 23444 means next-day carriage, while 23099 is a carriage lantern. PADS describes cushion pads. Ambiguous cases stay eligible unless there is evidence for excluding the whole code.

## 用训练期证据冻结商品编码规则

回查 2010 年 12 月 1 日至 2011 年 9 月 30 日的描述原文。登记表列出原始行号，以及该期间基础过滤前后的记录数，下方重新核对这些数值。使用区分大小写的完整编码。例如，23444 记录次日配送费，23099 是商品名称中带有 carriage 的灯，PADS 对应靠垫。对于依据不足的编码，继续按一般购买条件判断。

In [8]:
_POLICY = {
    "delivery": (
        "exclude", "Postage or carriage pays for delivery.",
        "邮费或运费支付的是配送服务。",
    ),
    "packing_fee": (
        "exclude", "The line bills a packing fee.",
        "该行收取包装费。",
    ),
    "financial_fee": (
        "exclude", "The description identifies a payment, platform or commission fee.",
        "描述表明该行记录银行、平台费用或佣金。",
    ),
    "bad_debt": (
        "exclude", "The line records a bad-debt adjustment.",
        "该行属于坏账调整。",
    ),
    "discount": (
        "exclude", "The line records a discount on a transaction.",
        "该行记录交易折扣。",
    ),
    "manual": (
        "exclude", "The source description only says Manual. It identifies no particular product to rank.",
        "原文描述只有 Manual，无法据此识别可供推荐的具体商品。",
    ),
    "sample": (
        "exclude", "The generic sample code does not identify a particular item to recommend.",
        "样品通用编码未指明可推荐的具体商品。",
    ),
    "voucher": (
        "exclude", "The denomination identifies prepaid store credit. This project ranks merchandise.",
        "面额对应店铺预付购物额度；本项目的推荐对象是商品。",
    ),
    "product": (
        "retain", "The training description names a physical product.",
        "训练期描述指明了具体实物商品。",
    ),
    "unresolved": (
        "retain", "Training rows contain only a missing description or a channel name. "
                  "Evidence is insufficient for a code-wide exclusion.",
        "训练期描述缺失或只记录销售渠道，没有足够证据对整个商品编码设定排除规则。",
    ),
}

# Columns: exact code, policy, literal example description, Excel row, time,
# all raw training rows, and training rows surviving the basic purchase filters.
_TRAINING_EVIDENCE = (
    ("23444", "delivery", "Next Day Carriage", 205750, "2011-05-26 16:11", 38, 36),
    ("23574", "packing_fee", "PACKING CHARGE", 252607, "2011-07-06 13:04", 9, 7),
    ("AMAZONFEE", "financial_fee", "AMAZON FEE", 14516, "2010-12-07 12:41", 27, 0),
    ("B", "bad_debt", "Adjust bad debt", 299984, "2011-08-12 14:50", 3, 0),
    ("BANK CHARGES", "financial_fee", "Bank Charges", 4408, "2010-12-02 15:08", 27, 7),
    ("C2", "delivery", "CARRIAGE", 1425, "2010-12-01 14:05", 102, 94),
    ("CRUK", "financial_fee", "CRUK Commission", 317510, "2011-08-30 10:49", 6, 0),
    ("D", "discount", "Discount", 143, "2010-12-01 09:41", 61, 0),
    ("DOT", "delivery", "DOTCOM POSTAGE", 1816, "2010-12-01 14:32", 603, 6),
    ("M", "manual", "Manual", 2241, "2010-12-01 15:35", 411, 188),
    ("m", "manual", "Manual", 40385, "2010-12-21 15:18", 1, 0),
    ("POST", "delivery", "POSTAGE", 47, "2010-12-01 08:45", 872, 770),
    ("S", "sample", "SAMPLES", 14438, "2010-12-07 12:03", 55, 0),
    ("gift_0001_10", "voucher", "Dotcomgiftshop Gift Voucher £10.00", 112444, "2011-03-07 17:14", 8, 0),
    ("gift_0001_20", "voucher", "Dotcomgiftshop Gift Voucher £20.00", 44796, "2011-01-05 14:44", 9, 0),
    ("gift_0001_30", "voucher", "Dotcomgiftshop Gift Voucher £30.00", 44727, "2011-01-05 14:44", 8, 0),
    ("gift_0001_40", "voucher", "Dotcomgiftshop Gift Voucher £40.00", 38250, "2010-12-20 10:14", 3, 0),
    ("gift_0001_50", "voucher", "Dotcomgiftshop Gift Voucher £50.00", 42059, "2010-12-23 13:26", 4, 0),
    ("21518", "product", "BANK ACCOUNT  GREETING  CARD ", 3343, "2010-12-02 10:39", 39, 17),
    ("23099", "product", "FRENCH CARRIAGE LANTERN", 191482, "2011-05-16 14:30", 42, 39),
    ("85123A", "product", "WHITE HANGING HEART T-LIGHT HOLDER", 2, "2010-12-01 08:26", 1813, 1581),
    ("PADS", "product", "PADS TO MATCH ALL CUSHIONS", 157197, "2011-04-15 09:27", 4, 3),
    ("DCGS0003", "product", "BOXED GLASS ASHTRAY", 24908, "2010-12-10 14:59", 5, 0),
    ("DCGS0004", "product", "HAYNES CAMPER SHOULDER BAG", 170785, "2011-04-27 17:23", 1, 0),
    ("DCGS0055", "unresolved", None, 74827, "2011-01-28 13:08", 1, 0),
    ("DCGS0057", "unresolved", None, 75297, "2011-01-28 15:01", 1, 0),
    ("DCGS0066P", "unresolved", None, 279259, "2011-07-26 11:52", 1, 0),
    ("DCGS0067", "unresolved", "ebay", 279258, "2011-07-26 11:52", 1, 0),
    ("DCGS0068", "unresolved", "ebay", 279257, "2011-07-26 11:52", 1, 0),
    ("DCGS0069", "product", "OOH LA LA DOGS COLLAR", 75055, "2011-01-28 14:25", 2, 0),
    ("DCGS0070", "product", "CAMOUFLAGE DOG COLLAR", 40054, "2010-12-21 13:06", 2, 0),
    ("DCGS0071", "unresolved", None, 279254, "2011-07-26 11:51", 1, 0),
    ("DCGS0072", "unresolved", None, 74840, "2011-01-28 13:08", 1, 0),
    ("DCGS0073", "unresolved", "ebay", 279253, "2011-07-26 11:51", 1, 0),
    ("DCGS0074", "unresolved", None, 74841, "2011-01-28 13:09", 1, 0),
    ("DCGS0076", "product", "SUNJAR LED NIGHT NIGHT LIGHT", 21328, "2010-12-09 14:09", 2, 0),
    ("DCGSSBOY", "product", "BOYS PARTY BAG", 84018, "2011-02-07 14:04", 10, 0),
    ("DCGSSGIRL", "product", "GIRLS PARTY BAG", 84019, "2011-02-07 14:04", 11, 0),
)

def merchandise_rules() -> pd.DataFrame:
    """Return the frozen exclusion decisions and reviewed retained codes with evidence."""
    return pd.DataFrame.from_records(
        [
            (code, *_POLICY[policy], *evidence)
            for code, policy, *evidence in _TRAINING_EVIDENCE
        ],
        columns=[
            "StockCode", "action", "reason_en", "reason_zh", "training_description",
            "training_example_excel_row", "training_example_date", "training_rows",
            "training_basic_rows",
        ],
    )

In [9]:
registry = merchandise_rules()
train_start = pd.Timestamp("2010-12-01")
cutoff = pd.Timestamp("2011-10-01")
training = raw.loc[raw["InvoiceDate"].ge(train_start) & raw["InvoiceDate"].lt(cutoff)]
basic = raw.loc[~pd.DataFrame(flags).any(axis=1)]
basic_training = basic.loc[basic["InvoiceDate"].ge(train_start) & basic["InvoiceDate"].lt(cutoff)]
evidence = raw.set_index("ExcelRow").loc[registry["training_example_excel_row"]]
assert evidence["StockCode"].tolist() == registry["StockCode"].tolist()
assert evidence["InvoiceDate"].tolist() == pd.to_datetime(registry["training_example_date"]).tolist()
assert evidence["Description"].fillna("").tolist() == registry["training_description"].fillna("").tolist()
for lines, field in [(training, "training_rows"), (basic_training, "training_basic_rows")]:
    counts = lines.groupby("StockCode").size()
    assert registry["StockCode"].map(counts).fillna(0).tolist() == registry[field].tolist()
with pd.option_context("display.max_rows", None):
    display(registry[["StockCode", "action", "training_description", "training_example_excel_row",
                      "training_rows", "training_basic_rows"]])

,StockCode,action,training_description,training_example_excel_row,training_rows,training_basic_rows
0,23444,exclude,Next Day Carriage,205750,38,36
1,23574,exclude,PACKING CHARGE,252607,9,7
2,AMAZONFEE,exclude,AMAZON FEE,14516,27,0
3,B,exclude,Adjust bad debt,299984,3,0
4,BANK CHARGES,exclude,Bank Charges,4408,27,7
5,C2,exclude,CARRIAGE,1425,102,94
6,CRUK,exclude,CRUK Commission,317510,6,0
7,D,exclude,Discount,143,61,0
8,DOT,exclude,DOTCOM POSTAGE,1816,603,6
9,M,exclude,Manual,2241,411,188


## Apply one decision per excluded code

Across the full workbook 571 uppercase M lines and one lowercase m line carry only the description Manual. The underlying item is unidentified, including both lines of invoice 561916. B has three entries explicitly labelled Adjust bad debt, including one positive-price entry. Keep every code’s raw count, even when earlier rules removed all its lines. The code register supplies the reason and a source row for each exclusion.

## 每个排除编码单设一步

全表有 571 条大写 M 和 1 条小写 m，描述均只有 Manual。发票 561916 的两行也只能看到这个标签，具体商品或费用无法确定。B 则有三条明确写着 Adjust bad debt 的账务记录，其中一条单价为正。即使较早的步骤已排除某个编码的全部记录，仍列出该编码在原表的命中数，并在登记表中保留排除理由及证据行号。

In [10]:
start = len(records)
frozen = merchandise_rules()
code_sheets = {
    "23444": "C_invoices, P_nonpositive", "23574": "C_invoices, P_nonpositive",
    "AMAZONFEE": "C_invoices, note_summary", "B": "P_nonpositive, case_examples, note_summary",
    "BANK CHARGES": "C_invoices, note_summary", "C2": "C_invoices, P_nonpositive, note_summary",
    "CRUK": "C_invoices, note_summary", "D": "C_invoices, case_examples, note_summary",
    "DOT": "C_invoices, P_nonpositive, note_summary", "M": "C_invoices, P_nonpositive, note_summary",
    "POST": "C_invoices, P_nonpositive, note_summary", "S": "C_invoices, note_summary",
    "gift_0001_10": "P_nonpositive", "gift_0001_20": "P_nonpositive", "gift_0001_30": "P_nonpositive",
}
for item in frozen.loc[frozen["action"].eq("exclude")].itertuples(index=False):
    sources = ["merchandise_rules.csv"]
    if item.StockCode in code_sheets:
        sources.append(code_sheets[item.StockCode])
    exclude(f"non_merchandise_{item.StockCode}", frame["StockCode"].eq(item.StockCode),
            ", ".join(sources), item.reason_en, item.reason_zh)
show_steps(records[start:])

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,16,non_merchandise_23444,81,392692,79,392613,"merchandise_rules.csv, C_invoices, P_nonpositive",Postage or carriage pays for delivery.,邮费或运费支付的是配送服务。
1,17,non_merchandise_23574,17,392613,14,392599,"merchandise_rules.csv, C_invoices, P_nonpositive",The line bills a packing fee.,该行收取包装费。
2,18,non_merchandise_AMAZONFEE,34,392599,0,392599,"merchandise_rules.csv, C_invoices, note_summary","The description identifies a payment, platform or commission fee.",描述表明该行记录银行、平台费用或佣金。
3,19,non_merchandise_B,3,392599,0,392599,"merchandise_rules.csv, P_nonpositive, case_examples, note_summary",The line records a bad-debt adjustment.,该行属于坏账调整。
4,20,non_merchandise_BANK CHARGES,37,392599,12,392587,"merchandise_rules.csv, C_invoices, note_summary","The description identifies a payment, platform or commission fee.",描述表明该行记录银行、平台费用或佣金。
5,21,non_merchandise_C2,144,392587,133,392454,"merchandise_rules.csv, C_invoices, P_nonpositive, note_summary",Postage or carriage pays for delivery.,邮费或运费支付的是配送服务。
6,22,non_merchandise_CRUK,16,392454,0,392454,"merchandise_rules.csv, C_invoices, note_summary","The description identifies a payment, platform or commission fee.",描述表明该行记录银行、平台费用或佣金。
7,23,non_merchandise_D,77,392454,0,392454,"merchandise_rules.csv, C_invoices, case_examples, note_summary",The line records a discount on a transaction.,该行记录交易折扣。
8,24,non_merchandise_DOT,710,392454,16,392438,"merchandise_rules.csv, C_invoices, P_nonpositive, note_summary",Postage or carriage pays for delivery.,邮费或运费支付的是配送服务。
9,25,non_merchandise_M,571,392438,279,392159,"merchandise_rules.csv, C_invoices, P_nonpositive, note_summary",The source description only says Manual. It identifies no particular product to rank.,原文描述只有 Manual，无法据此识别可供推荐的具体商品。


## Reconcile the baseline before credit allocation

Each original Excel row has one first decision in the gross positive-sale screen. The five-rule baseline leaves 392,692 lines. Merchandise filtering then leaves 391,060 known-customer, positive-price merchandise sale lines before credit allocation. Preserve original prices, including the three £0.001 PADS purchases. This candidate count is the starting denominator for measuring which lines lose all or part of their quantity later.

## 核对贷项分配前的销售基线

每个原始 Excel 行在候选正向销售筛选中都有唯一首条决定。五步基线剩 392,692 行；再筛选商品后，得到 391,060 条有客户编号的正价商品销售明细，尚未分配贷项。保留原始单价，包括 3 笔 £0.001 的 PADS 购买。用这些候选行作为起点，后面再核对哪些行被全额取消，哪些只减少部分件数。

In [11]:
retained = int(surviving.sum())
records.append({
    "step": len(records) + 1, "rule": "retained", "action": "retain",
    "raw_flagged": retained, "rows_before": retained, "rows_removed": 0,
    "rows_after": retained, "evidence_sheets": "filter_attribution, sku_names, merchandise_rules.csv",
    "reason_en": "Keep identified, dated, positive-quantity, positive-price merchandise lines. Preserve exact SKU and invoice IDs, including suffixes.",
    "reason_zh": "保留客户、时间、发票与商品明确且数量和单价均为正的商品明细，按原值保留发票及商品编码（含后缀）。",
})
decisions = pd.DataFrame({"ExcelRow": frame["ExcelRow"], "first_rule": first_rule})
purchases = frame.loc[surviving].copy()
purchases["line_amount_gbp"] = purchases["Quantity"] * purchases["UnitPrice"]
steps = pd.DataFrame.from_records(records)
assert len(decisions) == len(raw) == len(purchases) + steps["rows_removed"].sum()
removed = steps.loc[steps["action"].eq("exclude")].set_index("rule")
assert decisions["first_rule"].value_counts().reindex(removed.index, fill_value=0).eq(removed["rows_removed"]).all()
assert decisions.loc[decisions["first_rule"].eq("retained"), "ExcelRow"].tolist() == purchases["ExcelRow"].tolist()
display(steps[["step", "rule", "raw_flagged", "rows_removed", "rows_after"]])

,step,rule,raw_flagged,rows_removed,rows_after
0,1,exact_duplicate,5268,5268,536641
1,2,missing_customer_id,135080,135037,401604
2,3,missing_invoice_no,0,0,401604
3,4,missing_stock_code,0,0,401604
4,5,missing_invoice_date,0,0,401604
5,6,missing_quantity,0,0,401604
6,7,missing_unit_price,0,0,401604
7,8,cancelled_invoice,9288,8872,392732
8,9,negative_quantity,10624,0,392732
9,10,zero_quantity,0,0,392732


## Trace baseline exclusions to the field evidence

This cross-tabulation explains which eligibility rule was reached *first*. For instance, a non-C negative row can fail the customer check before the quantity check. An affected row can belong to several evidence groups. Within one group the rule columns sum to its raw row count. Below we also classify the same rows for overall accounting, where unknown customer IDs are allowed.

## 将候选销售的首条剔除规则追溯到原表

下表按“首次命中”归因。例如一条非 C 负数量行可能先在客户编号步骤被排除。原始行可以同时属于多个证据组，只有同一组内各规则的行数可以加总为该组的原始行数。下文另按整体经营账本口径处理这些行：账本允许客户编号缺失。

In [12]:
evidence_groups = {
    "C_invoices · C prefix": flags["cancelled_invoice"],
    "nonC_negative · negative quantity": ~flags["cancelled_invoice"] & raw["Quantity"].lt(0),
    "P_nonpositive · price <= 0": raw["UnitPrice"].le(0),
    "missingness · no description": raw["Description"].isna(),
    "P_nonpositive / note_summary · M or m": raw["StockCode"].isin(["M", "m"]),
    "case_examples · B / Adjust bad debt": raw["StockCode"].eq("B"),
    "P_nonpositive · same invoice & SKU mixed prices / products": zero_price["zero_price_same_item"],
}
reason_counts = {}
for label, mask in evidence_groups.items():
    counts = decisions.loc[mask, "first_rule"].value_counts()
    assert counts.sum() == mask.sum()
    reason_counts[label] = counts
crosswalk = pd.DataFrame(reason_counts).fillna(0).astype(int).T
crosswalk.insert(0, "raw_rows", crosswalk.sum(axis=1))
display(crosswalk)

first_rule,raw_rows,cancelled_invoice,exact_duplicate,missing_customer_id,non_merchandise_M,zero_price_manual,zero_price_no_priced_companion,zero_price_other_item,zero_price_same_item
C_invoices · C prefix,9288,8872,37,379,0,0,0,0,0
nonC_negative · negative quantity,1336,0,0,1336,0,0,0,0,0
P_nonpositive · price <= 0,2517,0,5,2472,0,6,6,23,5
missingness · no description,1454,0,0,1454,0,0,0,0,0
P_nonpositive / note_summary · M or m,572,175,5,107,279,6,0,0,0
case_examples · B / Adjust bad debt,3,0,0,3,0,0,0,0,0
P_nonpositive · same invoice & SKU mixed prices / products,23,0,0,18,0,0,0,0,5


## Build signed ledgers A and B

Starting from the original worksheet again, remove later exact duplicates, A-document bad-debt entries, inventory write-offs, ineligible prices/quantities and non-merchandise codes. Keep a merchandise sale with its positive quantity. Keep an eligible C merchandise credit with its original negative quantity and its own unit price. Ledger A covers all eligible lines, including anonymous transactions displayed as `Guest/Unknown`. Ledger B selects the same rows for identified customers only. A missing customer ID stays missing in the matching key.

For each ledger, signed recorded amount is the sum of original quantity × original unit price over its rows. A cancellation-adjusted gross purchase amount will use *remaining units at the original sale price*. Deducting a matched credit again from the signed ledger would double count it. The workbook contains invoice observations rather than bank payments. See `ledger_summary`, `sign_combinations`, `customer_id_invoices` and `sku_policy_differences`.

## 建立 A、B 两份带符号账本

重新从原表出发，去除重复副本、A 前缀坏账、库存核销、无效价量及非商品编码。符合条件的商品销售保留正数量，C 商品贷项保留原始负数量和该贷项自己的单价。A 是整体账本，匿名交易在展示时标为 `Guest/Unknown`；B 在此口径上仅保留客户编号已知的行。匹配键中的缺失客户编号始终保留缺失。

各账本按原始数量 × 本行单价求和，得到带符号的记录金额。稍后的“取消调整后毛购买金额”则按原销售行的剩余件数 × 原销售单价计算。带符号账本已含贷项，再扣一遍会重复冲减。原表只记录发票，无法核实到账。可对照 `ledger_summary`、`sign_combinations`、`customer_id_invoices` 与 `sku_policy_differences`。

In [13]:
import numpy as np


import pandas as pd


EUROPE_COUNTRIES = frozenset({
    "EIRE", "France", "Germany", "Netherlands", "Belgium", "Spain", "Switzerland",
    "Portugal", "Channel Islands", "Italy", "Norway", "Sweden", "Denmark",
    "Finland", "Austria", "Poland", "Cyprus", "Greece", "Czech Republic",
    "Malta", "Iceland", "European Community", "Lithuania",
})


SCENARIOS = {
    "five_digit_independent_strict": ("five_digit", False, False),
    "five_digit_independent_inclusive": ("five_digit", False, True),
    "frozen_independent_inclusive": ("frozen", False, True),
    "frozen_balance_strict": ("frozen", True, False),
    "frozen_balance_inclusive": ("frozen", True, True),
}


CREDIT_STATUSES = ("cancellation", "return", "unmatched", "anonymous", "unknown_region")


def _ledger_decisions(raw, stock_policy):
    """Apply mutually exclusive first-match reasons in original worksheet order."""
    if raw["ExcelRow"].isna().any() or not raw["ExcelRow"].is_unique:
        raise ValueError("ExcelRow must be present and unique in the raw worksheet.")
    frame = raw.sort_values("ExcelRow").reset_index(drop=True)
    code = frame["StockCode"].astype("string")
    if stock_policy == "frozen":
        rules = merchandise_rules()
        merchandise = ~code.isin(rules.loc[rules["action"].eq("exclude"), "StockCode"])
    elif stock_policy == "five_digit":
        merchandise = code.str.match(r"^[0-9]{5}", na=False)
    else:
        raise ValueError("stock_policy must be 'frozen' or 'five_digit'.")
    invoice = frame["InvoiceNo"].astype("string")
    credit = invoice.str.startswith("C", na=False)
    quantity = frame["Quantity"]
    price = frame["UnitPrice"]
    checks = [
        ("exact_duplicate", frame.duplicated(RAW_COLUMNS)),
        ("missing_required_field", frame[["InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice"]].isna().any(axis=1)),
        ("bad_debt_document", invoice.str.startswith("A", na=False)),
        ("non_c_negative_quantity", ~credit & quantity.lt(0)),
        ("zero_quantity", quantity.eq(0)),
        ("negative_unit_price", price.lt(0)),
        ("zero_unit_price", price.eq(0)),
        ("excluded_stock_code", ~merchandise),
        ("unexpected_positive_c_line", credit & quantity.gt(0)),
        ("credit", credit & quantity.lt(0)),
    ]
    decisions = np.select(
        [test.to_numpy(dtype=bool, na_value=False) for _, test in checks],
        [reason for reason, _ in checks],
        default="sale",
    )
    return frame, pd.Series(decisions, index=frame.index, name="decision")


def _signed_lines(frame, decisions):
    eligible = decisions.isin(["sale", "credit"])
    ledger = frame.loc[eligible].copy()
    ledger["ledger_type"] = decisions.loc[eligible]
    # Guest is a display value only. CustomerID stays missing for all joins.
    ledger["customer_display"] = ledger["CustomerID"].astype("string").fillna("Guest/Unknown")
    ledger["signed_amount_gbp"] = ledger["Quantity"] * ledger["UnitPrice"]
    return ledger.reset_index(drop=True)


def build_signed_ledger(raw: pd.DataFrame, stock_policy="frozen") -> pd.DataFrame:
    """Build store-level signed view A. View B selects its identified customers."""
    frame, decisions = _ledger_decisions(raw, stock_policy)
    return _signed_lines(frame, decisions)
ledger_preview = build_signed_ledger(raw)
signed_preview = ledger_preview.groupby(["ledger_type", ledger_preview["CustomerID"].notna().rename("identified")]).agg(
    rows=("ExcelRow", "size"), units=("Quantity", "sum"), signed_gbp=("signed_amount_gbp", "sum")
)
display(signed_preview)

rows    units   signed_gbp
ledger_type identified                              
credit      False          162    -2306    -4150.450
            True          8503  -267708  -471720.710
sale        False       131387   419600  1509991.680
            True        391060  5141694  8735922.643

## Allocate each credit once to an eligible sale

1. Process eligible C lines by credit timestamp and then by Excel row. A known customer and exact SKU must match. Among sales at or before that timestamp, choose the most recent single line with remaining reversible units at least equal to the magnitude of the credit. Enforce the unit balance across successive credits. A return also reserves its units. A partial credit can use part of one sale; combining several smaller sales would invent a linkage.
2. At equal candidate timestamps, prefer the lower Excel row, so an earlier sheet row wins if eligible. A later sheet row in the same minute can be selected when it is the eligible matching sale. Record both flags. A credit can have a different price from its sale.
3. Classify from the C row's country and actual elapsed time: UK up to and including 72 hours, the 23 named European destinations including EIRE up to 120 hours, and remaining known destinations up to 240 hours. Beyond the applicable threshold, assign the operational return label. Keep a missing or unspecified region, a known customer without an eligible match, and an anonymous credit separate. UCI identifies C as cancellation documentation; shipping, delivery, and return reasons are unobserved. [UCI dataset](https://doi.org/10.24432/C5BW33)

## 为每条商品贷项分配一个有余额的销售行

1. 按贷项时间、Excel 行号顺序处理。匹配同一已知客户、完全相同的商品编码，从贷项时刻或之前的销售中，选择时间最近且**单行可逆余额足够**的行。每次匹配都占用余额，包括判为退货的匹配。一次部分贷项可以使用一笔销售的部分件数；无法用多个小额销售行拼成一次匹配。
2. 同一时间有多条合格销售时取较小的 Excel 行号，因此贷项之前的合格行优先；只有同分钟靠后的行合格时也可以匹配，并记录同刻、行号靠后两个标记。配对无需两笔单价相同。
3. 按 C 行的国家和实际经过的时间判定：英国在 72 小时内（含边界），包括 EIRE 的 23 个列明欧洲地区在 120 小时内，其余已知地区在 240 小时内，归为取消；超过相应阈值归为操作性退货。地区缺失或未指定、客户已知但无可配销售、匿名贷项分别记录。UCI 只说明 C 是取消类单据，没有发货、签收时间和退货原因。[UCI 数据说明](https://doi.org/10.24432/C5BW33)

In [14]:
def _region(country):
    """Map the C document's destination to its continuous elapsed-time threshold."""
    if pd.isna(country) or country in ("", "Unspecified"):
        return "unknown", None
    if country == "United Kingdom":
        return "UK", 72
    if country in EUROPE_COUNTRIES:
        return "Europe", 120
    return "ROW", 240


def match_reversals(ledger: pd.DataFrame, *, use_balance=True, include_same_time=True) -> pd.DataFrame:
    """Match each C once to the nearest eligible single positive line.

    Work in credit timestamp and worksheet-row order. A sale must have the
    same original customer and SKU, with enough reversible units on that
    one row. Price is evidence but is not a matching condition. At the nearest
    eligible purchase time, ascending ExcelRow prefers rows before the C and
    then the smallest row. Inclusive matching allows an equal-time row appearing
    later on the sheet. Every match consumes balance in the primary policy,
    including a return or a match lacking usable country evidence. Only a
    classified cancellation reduces effective purchase history, when its C
    record has become observable.
    """
    credits = ledger.loc[ledger["ledger_type"].eq("credit")].sort_values(["InvoiceDate", "ExcelRow"])
    keys = ["CustomerID", "StockCode"]
    relevant = credits[keys].dropna().drop_duplicates()
    sales = ledger.loc[ledger["ledger_type"].eq("sale")].merge(relevant, on=keys, validate="many_to_one")
    # Newest eligible time first; the lower original row wins equal-time ties.
    sales = sales.sort_values(["InvoiceDate", "ExcelRow"], ascending=[False, True])
    groups = {}
    for sale in sales.itertuples(index=False):
        groups.setdefault((sale.CustomerID, sale.StockCode), []).append(sale)
    remaining = {}
    records = []
    for credit in credits.itertuples(index=False):
        customer, sku = credit.CustomerID, credit.StockCode
        units = -credit.Quantity
        region, hours = _region(credit.Country)
        record = {
            "credit_excel_row": credit.ExcelRow, "credit_invoice_no": credit.InvoiceNo,
            "credit_time": credit.InvoiceDate, "CustomerID": customer, "StockCode": sku,
            "Country": credit.Country, "region": region, "threshold_hours": hours,
            "credited_units": units, "credit_unit_price": credit.UnitPrice,
            "credit_amount_gbp": credit.signed_amount_gbp,
            "purchase_excel_row": None, "purchase_invoice_no": None, "purchase_time": None,
            "purchase_unit_price": None, "purchase_quantity": None, "balance_before": None,
            "balance_after": None, "lag_minutes": None, "lag_days": None,
            "same_time": False, "later_excel_row_same_time": False,
            "status": "anonymous" if pd.isna(customer) else "unmatched",
        }
        if pd.notna(customer):
            for sale in groups.get((customer, sku), ()):
                lag = credit.InvoiceDate - sale.InvoiceDate
                if lag < pd.Timedelta(0) or (lag == pd.Timedelta(0) and not include_same_time):
                    continue
                available = remaining.get(sale.ExcelRow, sale.Quantity)
                if available < units:
                    continue
                after = available - units
                if use_balance:
                    remaining[sale.ExcelRow] = after
                status = "unknown_region"
                if hours is not None:
                    status = "cancellation" if lag <= pd.Timedelta(hours=hours) else "return"
                record.update({
                    "purchase_excel_row": sale.ExcelRow, "purchase_invoice_no": sale.InvoiceNo,
                    "purchase_time": sale.InvoiceDate, "purchase_unit_price": sale.UnitPrice,
                    "purchase_quantity": sale.Quantity, "balance_before": available,
                    "balance_after": after, "lag_minutes": lag.total_seconds() / 60,
                    "lag_days": lag / pd.Timedelta(days=1), "same_time": lag == pd.Timedelta(0),
                    "later_excel_row_same_time": lag == pd.Timedelta(0) and sale.ExcelRow > credit.ExcelRow,
                    "status": status,
                })
                break
        records.append(record)
    columns = [
        "credit_excel_row", "credit_invoice_no", "credit_time", "CustomerID", "StockCode",
        "Country", "region", "threshold_hours", "credited_units", "credit_unit_price",
        "credit_amount_gbp", "purchase_excel_row", "purchase_invoice_no", "purchase_time",
        "purchase_unit_price", "purchase_quantity", "balance_before", "balance_after",
        "lag_minutes", "lag_days", "same_time", "later_excel_row_same_time", "status",
    ]
    result = pd.DataFrame.from_records(records, columns=columns)
    for name in ("credit_time", "purchase_time"):
        result[name] = pd.to_datetime(result[name])
    for name in ("credit_excel_row", "threshold_hours", "credited_units", "purchase_excel_row",
                 "purchase_quantity", "balance_before", "balance_after"):
        result[name] = result[name].astype("Int64")
    for name in ("credit_amount_gbp", "credit_unit_price", "purchase_unit_price", "lag_minutes", "lag_days"):
        result[name] = result[name].astype(float)
    for name in ("same_time", "later_excel_row_same_time"):
        result[name] = result[name].astype(bool)
    return result

## Reproduce the reported benchmark, then change one assumption at a time

The supplied comparison used a five-digit stock-code prefix and evaluated each credit against the original sale quantity independently. Under that comparison, accepting equal timestamps reproduces **8,668** credits: 1,481 cancellations, 5,913 returns, 1,112 known unmatched and 162 anonymous. Its A/B ledgers contain 531,172/399,656 rows. Those figures are comparison targets, not imposed totals.

Under the frozen merchandise register, three credits for numeric service codes 23444 and 23574 leave the eligible set, and the physical product PADS stays eligible. Enforcing balances changes 35 more credits to unmatched compared with independent matching under the same frozen catalog. The strict-time control further isolates the effect of accepting a sale recorded in the same minute. Inspect the scenario summaries and individual worksheet rows below. The totals in the printed controls are computed from the workbook.

## 复核讨论中的对照数据，再逐项改变假设

讨论中的对照采用“五位数字开头”筛选商品；每条贷项独立使用原销售件数作为可逆额度。该口径允许时间相等时，复现 **8,668 条**商品贷项：取消 1,481 条、退货 5,913 条、已知客户未匹配 1,112 条、匿名 162 条。对照的 A、B 账本分别为 531,172 行、399,656 行。这些数字用于比较，不充当筛选目标。

正式方案使用有训练期证据的商品编码表，剔除属于数字服务编码 23444 和 23574 的三条贷项，保留实物商品 PADS。在冻结编码口径下，与独立配对相比，使用剩余额度后另有 35 条转为未匹配。进一步比较严格早于和允许同刻两种方案，核查同一分钟候选行的影响。下方既给出各方案的全表汇总，也能定位到原始 Excel 行；对照结果均由工作簿实际算出。

In [15]:
def _scenario_summary(scenario, ledger, matches, use_balance):
    """Reconcile signed amounts at the prices recorded on each source document."""
    rows = []

    def record(metric, frame, units="Quantity", money="signed_amount_gbp"):
        rows.append({
            "scenario": scenario, "metric": metric, "rows": len(frame),
            "units": frame[units].sum(), "signed_amount_gbp": frame[money].sum(),
        })

    known = ledger.loc[ledger["CustomerID"].notna()]
    sales = ledger.loc[ledger["ledger_type"].eq("sale")]
    record("view_a", ledger)
    record("view_b", known)
    record("guest", ledger.loc[ledger["CustomerID"].isna()])
    record("sale", sales)
    record("credit", ledger.loc[ledger["ledger_type"].eq("credit")])
    record("known_observed_sales", sales.loc[sales["CustomerID"].notna()])
    statuses = matches.assign(signed_units=-matches["credited_units"])
    for status in CREDIT_STATUSES:
        selected = statuses.loc[statuses["status"].eq(status)]
        record(status, selected, units="signed_units", money="credit_amount_gbp")
    if use_balance:
        cancelled = matches.loc[matches["status"].eq("cancellation")]
        cancelled_units = cancelled.groupby("purchase_excel_row")["credited_units"].sum()
        goods = sales.copy()
        goods["effective_units"] = goods["Quantity"] - goods["ExcelRow"].map(cancelled_units).fillna(0)
        goods = goods.loc[goods["effective_units"].gt(0)].copy()
        goods["effective_amount_gbp"] = goods["effective_units"] * goods["UnitPrice"]
        record("gross_after_cancellations", goods, "effective_units", "effective_amount_gbp")
        record("known_gross_after_cancellations", goods.loc[goods["CustomerID"].notna()],
               "effective_units", "effective_amount_gbp")
    return pd.DataFrame(rows)


def _compare_matches(before_name, before, after_name, after):
    """Trace lines whose inclusion, selected source, classification or balance changed."""
    detail = ["status", "purchase_excel_row", "purchase_invoice_no", "purchase_time",
              "purchase_unit_price", "lag_minutes", "balance_before", "balance_after",
              "same_time", "later_excel_row_same_time"]
    ids = ["credit_excel_row", "credit_invoice_no", "credit_time", "CustomerID", "StockCode", "region",
           "credited_units", "credit_amount_gbp"]
    identity = pd.concat([before[ids], after[ids]], ignore_index=True).drop_duplicates("credit_excel_row")
    key = "credit_excel_row"
    side = before.set_index(key)[detail].add_prefix("before_").join(
        after.set_index(key)[detail].add_prefix("after_"), how="outer",
    )
    changes = {}
    for field in ["status", "purchase_excel_row", "balance_before", "balance_after"]:
        old, new = side["before_" + field], side["after_" + field]
        changes[field] = (old.isna() != new.isna()) | old.ne(new).fillna(False)
    side["status_changed"] = changes["status"]
    side["purchase_changed"] = changes["purchase_excel_row"]
    side["balance_changed"] = changes["balance_before"] | changes["balance_after"]
    relevant = side["status_changed"] | side["purchase_changed"] | side["balance_changed"]
    result = identity.merge(side.loc[relevant], on=key, how="inner")
    result.insert(0, "after_scenario", after_name)
    result.insert(0, "before_scenario", before_name)
    return result.sort_values(key, ignore_index=True)


def _reference_benchmark(summary, regional):
    """Compare supplied figures with measured outputs without fitting the policy to them."""
    targets = {
        "view_a": (531172, 9770919.71), "view_b": (399656, 8265476.93),
        "credit": (8668, None), "cancellation": (1481, -312072.49),
        "return": (5913, -134674.78), "unmatched": (1112, -25003.44),
        "anonymous": (162, -4150.45),
    }
    rows = []

    def add(scenario, scope, metric, expected_rows, expected_amount, observed_rows, observed_amount):
        rows.append({
            "scenario": scenario, "scope": scope, "metric": metric, "reported_rows": expected_rows,
            "observed_rows": observed_rows, "delta_rows": observed_rows - expected_rows,
            "reported_amount_gbp": expected_amount, "observed_amount_gbp": observed_amount,
            "delta_amount_gbp": None if expected_amount is None else observed_amount - expected_amount,
        })

    for row in summary.itertuples(index=False):
        if row.metric in targets:
            count, amount = targets[row.metric]
            add(row.scenario, "all", row.metric, count, amount, row.rows, row.signed_amount_gbp)
    # The supplied regional 'unmatched' totals include anonymous credits. The
    # main ledger retains the separate unmatched, anonymous and unknown-region states.
    region_targets = {"UK": (1111, 5084, 1134), "Europe": (194, 788, 133), "ROW": (176, 41, 7)}
    for scenario in SCENARIOS:
        for region, counts in region_targets.items():
            group = regional.loc[regional["scenario"].eq(scenario) & regional["region"].eq(region)]
            for metric, statuses, reported in zip(
                ["cancellation", "return", "unmatched_including_anonymous"],
                [["cancellation"], ["return"], ["unmatched", "anonymous"]], counts,
            ):
                selected = group.loc[group["status"].isin(statuses)]
                add(scenario, region, metric, reported, None,
                    selected["rows"].sum(), selected["signed_amount_gbp"].sum())
    return pd.DataFrame(rows)

In [16]:
def compute_reversal_tables(raw: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Classify the original eight-field worksheet with its stable ExcelRow locators.

    Return canonical signed A/B, canonical credit allocations, a source audit and
    the benchmark controls. Raw worksheet counts, independently classified credits
    and balance-reserved credits have different denominators. Every comparison
    below preserves its own source policy, lag rule and allocation convention.
    """
    sources, decisions, audits = {}, {}, []
    for policy in ("five_digit", "frozen"):
        frame, decision = _ledger_decisions(raw, policy)
        decisions[policy] = decision
        sources[policy] = _signed_lines(frame, decision)
        audit = pd.DataFrame({
            "reason": decision, "customer_id_status": np.where(frame["CustomerID"].isna(), "anonymous", "known"),
            "signed_amount_gbp": frame["Quantity"] * frame["UnitPrice"],
        })
        audit = audit.groupby(["reason", "customer_id_status"], as_index=False).agg(
            rows=("reason", "size"), signed_amount_gbp=("signed_amount_gbp", "sum"),
        )
        audit.insert(0, "stock_policy", policy)
        audits.append(audit)

    difference = decisions["five_digit"].ne(decisions["frozen"])
    detail = frame.loc[difference].copy()
    detail["five_digit_decision"] = decisions["five_digit"].loc[difference]
    detail["frozen_decision"] = decisions["frozen"].loc[difference]
    detail["signed_amount_gbp"] = detail["Quantity"] * detail["UnitPrice"]

    matches, summaries, regions = {}, [], []
    for name, (policy, balance, same_time) in SCENARIOS.items():
        match = match_reversals(sources[policy], use_balance=balance, include_same_time=same_time)
        matches[name] = match
        summaries.append(_scenario_summary(name, sources[policy], match, balance))
        region = match.groupby(["region", "status"], as_index=False).agg(
            rows=("status", "size"), signed_amount_gbp=("credit_amount_gbp", "sum"),
        )
        region.insert(0, "scenario", name)
        regions.append(region)
    comparisons = [
        ("five_digit_independent_strict", "five_digit_independent_inclusive"),
        ("five_digit_independent_inclusive", "frozen_independent_inclusive"),
        ("frozen_independent_inclusive", "frozen_balance_inclusive"),
        ("frozen_balance_strict", "frozen_balance_inclusive"),
    ]
    changes = [
        _compare_matches(before, matches[before], after, matches[after]) for before, after in comparisons
    ]
    summary = pd.concat(summaries, ignore_index=True)
    region_summary = pd.concat(regions, ignore_index=True)
    signed = sources["frozen"]
    tables = {
        "signed_ledger_a": signed,
        "signed_ledger_b": signed.loc[signed["CustomerID"].notna()].reset_index(drop=True),
        "credit_allocations": matches["frozen_balance_inclusive"],
        "source_audit": pd.concat(audits, ignore_index=True),
        "source_differences": detail.reset_index(drop=True),
        "reconciliation": summary,
        "region_summary": region_summary,
        "match_differences": pd.concat(changes, ignore_index=True),
        "benchmark": _reference_benchmark(summary, region_summary),
    }
    # Keep sub-penny source prices. Remove floating-point summation noise from
    # published diagnostics without modifying source prices or the signed ledgers.
    for name in ["source_audit", "reconciliation", "region_summary", "benchmark"]:
        money = [column for column in tables[name] if column.endswith("_gbp")]
        rounded = tables[name][money].round(6)
        # A rounded zero has a single CSV representation across NumPy versions.
        tables[name][money] = rounded.mask(rounded.eq(0), 0.0)
    return tables
reversal_tables = compute_reversal_tables(raw)
ledger_a = reversal_tables["signed_ledger_a"]
ledger_b = reversal_tables["signed_ledger_b"]
allocations = reversal_tables["credit_allocations"]
assert len(ledger_preview) == len(ledger_a)
del ledger_preview
for view, ledger in [("A", ledger_a), ("B", ledger_b)]:
    print(f"{view}: {len(ledger):,} rows / 行 · £{ledger['signed_amount_gbp'].sum():,.3f}")
benchmark = reversal_tables["benchmark"]
display(benchmark.loc[benchmark["scope"].eq("all") & benchmark["scenario"].isin(
    ["five_digit_independent_inclusive", "frozen_balance_inclusive"]
), ["scenario", "metric", "reported_rows", "observed_rows", "delta_rows",
    "reported_amount_gbp", "observed_amount_gbp", "delta_amount_gbp"]])
scenario_counts = reversal_tables["reconciliation"].pivot(index="scenario", columns="metric", values="rows")
display(scenario_counts[["view_a", "view_b", "credit", "cancellation", "return", "unmatched", "anonymous"]])
regions = reversal_tables["region_summary"]
display(regions.loc[regions["scenario"].eq("frozen_balance_inclusive"),
                    ["region","status","rows","signed_amount_gbp"]])

A: 531,112 rows / 行 · £9,770,043.163
B: 399,563 rows / 行 · £8,264,201.933


,scenario,metric,reported_rows,observed_rows,delta_rows,reported_amount_gbp,observed_amount_gbp,delta_amount_gbp
7,five_digit_independent_inclusive,view_a,531172,531172,0,9770919.71,9770919.710,0.000
8,five_digit_independent_inclusive,view_b,399656,399656,0,8265476.93,8265476.930,0.000
9,five_digit_independent_inclusive,credit,8668,8668,0,NaN,-475901.160,NaN
10,five_digit_independent_inclusive,cancellation,1481,1481,0,-312072.49,-312072.490,0.000
11,five_digit_independent_inclusive,return,5913,5913,0,-134674.78,-134674.780,0.000
12,five_digit_independent_inclusive,unmatched,1112,1112,0,-25003.44,-25003.440,0.000
13,five_digit_independent_inclusive,anonymous,162,162,0,-4150.45,-4150.450,0.000
28,frozen_balance_inclusive,view_a,531172,531112,-60,9770919.71,9770043.163,-876.547
29,frozen_balance_inclusive,view_b,399656,399563,-93,8265476.93,8264201.933,-1274.997
30,frozen_balance_inclusive,credit,8668,8665,-3,NaN,-475871.160,NaN


metric,view_a,view_b,credit,cancellation,return,unmatched,anonymous
scenario,,,,,,,
five_digit_independent_inclusive,531172.0,399656.0,8668.0,1481.0,5913.0,1112.0,162.0
five_digit_independent_strict,531172.0,399656.0,8668.0,1476.0,5918.0,1112.0,162.0
frozen_balance_inclusive,531112.0,399563.0,8665.0,1473.0,5883.0,1147.0,162.0
frozen_balance_strict,531112.0,399563.0,8665.0,1466.0,5888.0,1149.0,162.0
frozen_independent_inclusive,531112.0,399563.0,8665.0,1479.0,5912.0,1112.0,162.0


,region,status,rows,signed_amount_gbp
48,Europe,anonymous,49,-1190.19
49,Europe,cancellation,193,-11480.38
50,Europe,return,780,-12167.58
51,Europe,unmatched,92,-2515.96
52,ROW,anonymous,2,-216.69
53,ROW,cancellation,176,-2311.48
54,ROW,return,41,-2702.88
55,ROW,unmatched,5,-255.19
56,UK,anonymous,111,-2743.57
57,UK,cancellation,1104,-298020.28


## Read the signed credit total and the reversible units separately

The canonical A ledger holds **531,112** rows and £9,770,043.163 signed. B holds **399,563** rows and £8,264,201.933 signed. The 8,665 eligible C lines consist of **1,473 cancellations** (−£311,812.14), **5,883 returns** (−£133,795.18), **1,147 unmatched known-customer credits** (−£26,113.39) and **162 anonymous credits** (−£4,150.45). Their amounts use the original C prices. The separate unknown-region category is empty on this workbook.

The cancellation matches remove 189,315 units worth £349,595.90 at *sale* prices. For sale 556444 and credit C556448, 60 units at £649.50 on the original sale are paired with 60 units at £4.95 on the credit. The gross purchase adjustment is £38,970; the signed ledger credit is −£297. The price discrepancy remains visible for review. The strict- and inclusive-time balance controls differ on seven classifications. Twelve primary matches have the same timestamp; the sheet exposes the two Excel row numbers.

## 分别核对带符号贷项金额与已分配件数

正式 A 账本为 **531,112 行，带符号金额 £9,770,043.163**；B 为 **399,563 行，£8,264,201.933**。8,665 条合格商品 C 行分为 **1,473 条取消**（−£311,812.14）、**5,883 条退货**（−£133,795.18）、**1,147 条已知客户未匹配**（−£26,113.39）、**162 条匿名贷项**（−£4,150.45）。金额均按 C 行自己的原始单价计算。本表中单列的地区未知匹配为零。

取消共减少 189,315 件，按原销售单价计算的购买金额减少 £349,595.90。例如发票 556444 的 60 件商品单价 £649.50，对应 C556448 冲回 60 件、贷项单价 £4.95。因此有效毛购买金额减少 £38,970，而带符号账本中的贷项是 −£297；单价不同的证据可直接回查。使用余额的两个时间对照中，严格早于和允许同刻有七条分类不同。正式匹配中共有十二条与销售同刻，每条都可查双方的 Excel 行号。

In [17]:
credit_cols = ["credit_excel_row", "credit_invoice_no", "purchase_excel_row", "purchase_invoice_no",
               "CustomerID", "StockCode", "purchase_quantity", "credited_units", "balance_before",
               "balance_after", "lag_minutes", "purchase_unit_price", "credit_unit_price", "status"]
credit_totals = allocations.groupby("status", dropna=False).agg(
    rows=("credit_excel_row", "size"), units=("credited_units", "sum"), signed_credit_gbp=("credit_amount_gbp", "sum"))
display(credit_totals)
display(allocations.loc[allocations["same_time"], credit_cols + ["later_excel_row_same_time"]])
display(allocations.loc[allocations["credit_invoice_no"].isin(["C541433", "C581484", "C556448"]), credit_cols])
different_skus = reversal_tables["source_differences"]
display(different_skus.groupby(["StockCode","five_digit_decision","frozen_decision"]).agg(
    lines=("ExcelRow","size"), example_excel_row=("ExcelRow","min"), signed_gbp=("signed_amount_gbp","sum")
).reset_index())
changed = reversal_tables["match_differences"]
display(changed.groupby(["before_scenario","after_scenario"])[["status_changed","purchase_changed","balance_changed"]].sum())
display(changed.loc[changed["status_changed"], ["before_scenario","after_scenario","credit_excel_row",
    "credit_invoice_no","before_purchase_excel_row","after_purchase_excel_row","before_status","after_status"]].head(12))

,rows,units,signed_credit_gbp
status,,,
anonymous,162,2306,-4150.45
cancellation,1473,189315,-311812.14
return,5883,58386,-133795.18
unmatched,1147,20007,-26113.39


,credit_excel_row,credit_invoice_no,purchase_excel_row,purchase_invoice_no,CustomerID,StockCode,purchase_quantity,credited_units,balance_before,balance_after,lag_minutes,purchase_unit_price,credit_unit_price,status,later_excel_row_same_time
139,13021,C537408,13015,537407,15502,47580,6,6,6,0,0.0,2.55,2.55,cancellation,False
140,13022,C537408,13020,537407,15502,22196,12,12,12,0,0.0,0.85,0.85,cancellation,False
1001,62714,C541512,62713,541513,17576,85036B,12,6,12,6,0.0,1.25,4.25,cancellation,False
1693,94833,C544429,94832,544430,16306,22637,2,2,2,0,0.0,2.55,2.55,cancellation,False
2138,125233,C547031,125206,547029,15465,22423,32,8,32,24,0.0,10.95,10.95,cancellation,False
2668,155039,C549962,155038,549961,16652,22423,2,2,2,0,0.0,12.75,10.95,cancellation,False
2836,164797,C550672,164796,550671,12457,23199,100,100,100,0,0.0,1.79,1.79,cancellation,False
6099,365082,C568694,365080,568695,16321,22090,12,12,12,0,0.0,2.95,2.95,cancellation,False
6100,365083,C568694,365081,568695,16321,21198,24,24,24,0,0.0,1.65,1.65,cancellation,False
7800,466264,C576324,466235,576323,16161,21259,2,1,2,1,0.0,5.95,5.95,cancellation,False


,credit_excel_row,credit_invoice_no,purchase_excel_row,purchase_invoice_no,CustomerID,StockCode,purchase_quantity,credited_units,balance_before,balance_after,lag_minutes,purchase_unit_price,credit_unit_price,status
985,61626,C541433,61621,541431,12346,23166,74215,74215,74215,0,16.0,1.04,1.04,cancellation
3730,222694,C556448,222682,556444,15098,22502,60,60,60,0,11.0,649.50,4.95,cancellation
8659,540424,C581484,540423,581483,16446,23843,80995,80995,80995,0,12.0,2.08,2.08,cancellation


,StockCode,five_digit_decision,frozen_decision,lines,example_excel_row,signed_gbp
0,23444,credit,excluded_stock_code,1,252024,-15.000
1,23444,sale,excluded_stock_code,79,205750,1200.000
2,23574,credit,excluded_stock_code,2,309379,-15.000
3,23574,sale,excluded_stock_code,14,252607,105.000
4,DCGS0003,excluded_stock_code,sale,4,24908,9.940
5,DCGS0004,excluded_stock_code,sale,1,170785,16.630
6,DCGS0069,excluded_stock_code,sale,1,75055,15.790
7,DCGS0070,excluded_stock_code,sale,1,40054,12.720
8,DCGS0076,excluded_stock_code,sale,2,21328,48.390
9,DCGSSBOY,excluded_stock_code,sale,11,84018,150.550


,,status_changed,purchase_changed,balance_changed
before_scenario,after_scenario,,,
five_digit_independent_inclusive,frozen_independent_inclusive,3,3,3
five_digit_independent_strict,five_digit_independent_inclusive,5,12,4
frozen_balance_strict,frozen_balance_inclusive,7,12,5
frozen_independent_inclusive,frozen_balance_inclusive,35,57,275


,before_scenario,after_scenario,credit_excel_row,credit_invoice_no,before_purchase_excel_row,after_purchase_excel_row,before_status,after_status
3,five_digit_independent_strict,five_digit_independent_inclusive,94833,C544429,87290,94832,return,cancellation
4,five_digit_independent_strict,five_digit_independent_inclusive,125233,C547031,110139,125206,return,cancellation
5,five_digit_independent_strict,five_digit_independent_inclusive,155039,C549962,117394,155038,return,cancellation
9,five_digit_independent_strict,five_digit_independent_inclusive,466264,C576324,147001,466235,return,cancellation
11,five_digit_independent_strict,five_digit_independent_inclusive,527316,C580702,464269,527315,return,cancellation
12,five_digit_independent_inclusive,frozen_independent_inclusive,252024,C559059,249752,<NA>,cancellation,NaN
13,five_digit_independent_inclusive,frozen_independent_inclusive,309379,C564084,305346,<NA>,cancellation,NaN
14,five_digit_independent_inclusive,frozen_independent_inclusive,374912,C569410,368927,<NA>,return,NaN
16,frozen_independent_inclusive,frozen_balance_inclusive,13011,C537406,9784,<NA>,cancellation,unmatched
17,frozen_independent_inclusive,frozen_balance_inclusive,13012,C537406,9783,<NA>,cancellation,unmatched


## Replay completed baskets and credit timestamps

A credit creates no positive event. In a full-period or as-of snapshot, take completed customer invoices and cancellations effective before the exclusive cutoff. Subtract cancelled units at the original unit price. Keep positive residuals, including partial lines (e.g. 10 sold, 3 cancelled leaves 7). A matched return reserves its allocation and leaves the original purchase history intact; it remains a signed credit for net accounting. Unknown-region, unmatched and anonymous credits do not reduce an uncertain purchase.

Each item event is unique per customer, invoice and SKU. Sum amounts computed at each retained line's own price, keeping fractional pennies. Use the invoice's original eligible-sale completion time even if its latest line is fully reversed. Features for D2 must replay the ledger at each history cutoff; labels use credits observable by the target window's end. A later October/November/December credit must never revise the October 1 training snapshot.

## 按购物篮完成时刻与贷项生效时刻回放

贷项本身不增加正向购买事件。对全期或指定截止点的快照，取已完成的客户发票，只应用严格早于截止点的取消。按原销售单价扣除相应件数，部分取消后仍有余量便保留（例如售出 10 件、取消 3 件，保留 7 件）。匹配退货会占用可逆额度，原购买仍进入历史；在净额口径中该贷项照常冲减。地区未知、未匹配和匿名贷项均不冲减无法确定的购买。

每个客户、发票、商品对应一个购买事件，每行先按自身单价计额再相加，保留小于一便士的价格精度。即使最后一条原始合格销售被全额取消，仍保留原购物篮完成时间。D2 对每个历史截止点重新回放，标签只使用目标窗口结束前可见的贷项。十月、十一月、十二月出现的新贷项不能回改十月一日前的训练快照。

In [18]:
import pandas as pd


INVOICE_KEYS = ["CustomerID", "InvoiceNo"]


ITEM_KEYS = [*INVOICE_KEYS, "StockCode"]


def build_basket_events(purchases: pd.DataFrame) -> pd.DataFrame:
    """Aggregate each item within a customer invoice at the basket's completion time."""
    # Use the original row order for reproducible sums, including fractional pennies.
    ordered = purchases.sort_values("ExcelRow")
    # A reversed or reduced line may have carried the invoice's last timestamp.
    # Snapshot lines preserve the completion time of the original eligible basket.
    time_column = "basket_time" if "basket_time" in ordered.columns else "InvoiceDate"
    events = ordered.groupby(ITEM_KEYS, as_index=False, sort=False).agg(
        basket_time=(time_column, "max"),
        quantity=("Quantity", "sum"),
        amount_gbp=("line_amount_gbp", "sum"),
        source_lines=("ExcelRow", "size"),
    )
    events["basket_time"] = events.groupby(INVOICE_KEYS, sort=False)["basket_time"].transform("max")
    columns = ["CustomerID", "InvoiceNo", "basket_time", "StockCode", "quantity", "amount_gbp", "source_lines"]
    return events.sort_values(["CustomerID", "basket_time", "InvoiceNo", "StockCode"], ignore_index=True)[columns]


def training_basket_counts(purchases: pd.DataFrame, cutoff: pd.Timestamp) -> dict[str, int]:
    """Count multi-time invoices and multi-price items in baskets completed before cutoff."""
    if "basket_time" in purchases.columns:
        completion = purchases["basket_time"]
    else:
        completion = purchases.groupby(INVOICE_KEYS, sort=False)["InvoiceDate"].transform("max")
    history = purchases.loc[completion.lt(cutoff)]
    invoice_times = history.groupby(INVOICE_KEYS, sort=False)["InvoiceDate"].nunique()
    item_prices = history.groupby(ITEM_KEYS, sort=False)["UnitPrice"].nunique()
    return {
        "multi_time_invoices": int(invoice_times.gt(1).sum()),
        "multi_price_invoice_items": int(item_prices.gt(1).sum()),
    }

In [19]:
import pandas as pd


def purchase_line_balances(ledger, allocations, *, cutoff=None):
    """Return all identified positive source lines in completed baskets, including fully cancelled ones.

    Both inputs must use canonical allocation results and Excel worksheet row
    identifiers. For historical replay, pass a cutoff. Only baskets completed
    and credit classifications effective strictly before that timestamp enter.
    None requests a full-workbook retrospective view. The immutable invoice
    completion is computed before subtracting any matched cancellations.
    """
    sale = ledger["ledger_type"].eq("sale") & ledger["CustomerID"].notna()
    lines = ledger.loc[sale, ["ExcelRow", *RAW_COLUMNS]].copy()
    lines["InvoiceDate"] = pd.to_datetime(lines["InvoiceDate"], format="mixed")
    lines["basket_time"] = lines.groupby(INVOICE_KEYS, sort=False)["InvoiceDate"].transform("max")
    effective = allocations.loc[allocations["status"].isin(["cancellation", "return"])]
    if cutoff is not None:
        end = pd.Timestamp(cutoff)
        lines = lines.loc[lines["basket_time"].lt(end)].copy()
        credit_dates = pd.to_datetime(effective["credit_time"], format="mixed")
        effective = effective.loc[credit_dates.lt(end)]

    lines["original_quantity"] = lines["Quantity"]
    for status, column in [("cancellation", "cancelled_units"), ("return", "returned_units")]:
        totals = (
            effective.loc[effective["status"].eq(status)]
            .groupby("purchase_excel_row")["credited_units"].sum()
        )
        lines[column] = lines["ExcelRow"].map(totals).fillna(0).astype("int64")
    if (lines["cancelled_units"] + lines["returned_units"] > lines["original_quantity"]).any():
        raise ValueError("Matched credit units exceed the original quantity on a sale line.")
    lines["Quantity"] = lines["original_quantity"] - lines["cancelled_units"]
    lines["line_amount_gbp"] = lines["Quantity"] * lines["UnitPrice"]
    return lines.sort_values("ExcelRow", ignore_index=True)


def purchase_snapshot(ledger, allocations, *, cutoff=None):
    """Return surviving positive purchase lines and corresponding basket-item events."""
    balances = purchase_line_balances(ledger, allocations, cutoff=cutoff)
    purchases = balances.loc[balances["Quantity"].gt(0)].copy()
    return purchases, build_basket_events(purchases)
train_end = pd.Timestamp("2011-10-01")
full_balances = purchase_line_balances(ledger_b, allocations)
full_lines, full_events = purchase_snapshot(ledger_b, allocations)
train_lines, train_events = purchase_snapshot(ledger_b, allocations, cutoff=train_end)
import math
for valid, baskets in [(full_lines, full_events), (train_lines, train_events)]:
    assert not baskets.duplicated(ITEM_KEYS).any()
    assert baskets["source_lines"].sum() == len(valid)
    assert baskets["quantity"].sum() == valid["Quantity"].sum()
    assert math.isclose(baskets["amount_gbp"].sum(), valid["line_amount_gbp"].sum(), rel_tol=1e-12)
views = {"full_window": (full_lines, full_events), "before_2011-10-01": (train_lines, train_events)}
display(pd.DataFrame([{"view":name,"positive_lines":len(frame),"basket_item_events":len(events),
                        "baskets":len(events[INVOICE_KEYS].drop_duplicates()),"effective_units":frame["Quantity"].sum(),
                        "effective_gross_gbp":frame["line_amount_gbp"].sum()}
                       for name,(frame,events) in views.items()]))

,view,positive_lines,basket_item_events,baskets,effective_units,effective_gross_gbp
0,full_window,390168,385352,18331,4952379,8386326.743
1,before_2011-10-01,262067,259676,13040,3514634,5935091.803


## Follow quantities from invoice lines to basket events

In the full window **892 original sale lines reach zero** and **568 are partially cancelled**. The result contains **390,168** surviving positive lines and **385,352** invoice–item events. The October 1 snapshot contains **262,067** positive lines and **259,676** events. Inspect the two very large purchases against C541433 and C581484: 74,215 and 80,995 units were cancelled after 16 and 12 minutes, respectively. Removing only their C documents would leave £245,653.20 in spurious positive purchases. Both disappear from the effective full-period baskets; the December pair lies outside the training period. Large unreversed wholesale quantities remain.

Compare invoice 550188: the paid line of eight units × £7.65 contributes £61.20; the separate zero-price unit contributes no paid units. Invoice 539318 has 24 units × £0.55 and 120 × £0.42 of SKU 21213, giving one basket event with 144 units and £63.60. The partial-line and return samples below show how original quantity, cancelled units and returned units are recorded separately.

## 从发票明细追踪件数直到购物篮

全观察期**有 892 条原销售明细被全额取消，568 条部分取消**。最终保留 **390,168** 条有余量的正向明细、**385,352** 个发票–商品事件。十月一日快照有 **262,067** 条有效明细与 **259,676** 个事件。两笔巨额销售分别对应 C541433 和 C581484，74,215 件和 80,995 件分别在 16 和 12 分钟后被全额取消。只剔除 C 行会虚增 £245,653.20 的正向购买。全期有效购物篮中两笔均不再出现；十二月的一笔也在训练期之外。未被取消的大额批发仍保留。

对照发票 550188，8 件正价商品 × £7.65 计 £61.20，另外 1 件零价不增加计价件数。发票 539318 的商品 21213 分两行记录，24 × £0.55 加 120 × £0.42，共形成一次购买、144 件、£63.60。下方的部分取消和退货样例可分别核查原件数、已取消件数和退回件数。

In [20]:
sale_cols = ["ExcelRow", "CustomerID", "InvoiceNo", "StockCode", "basket_time", "original_quantity",
             "cancelled_units", "returned_units", "Quantity", "UnitPrice", "line_amount_gbp"]
huge = full_balances.loc[full_balances["InvoiceNo"].isin(["541431", "581483"])]
display(huge[sale_cols])
assert set(huge["original_quantity"]) == {74215, 80995} and huge["Quantity"].eq(0).all()
assert not full_events["InvoiceNo"].isin(["541431", "581483"]).any()
assert not train_events["InvoiceNo"].isin(["541431", "581483"]).any()
show_cases = [("550188", "22636"), ("539318", "21213")]
display(raw.loc[pd.MultiIndex.from_frame(raw[["InvoiceNo","StockCode"]]).isin(show_cases),
                ["ExcelRow", "InvoiceNo", "StockCode", "Quantity", "UnitPrice"]])
display(full_events.loc[pd.MultiIndex.from_frame(full_events[["InvoiceNo","StockCode"]]).isin(show_cases)])
display(full_balances.loc[full_balances["cancelled_units"].gt(0) & full_balances["Quantity"].gt(0),sale_cols].head(3))
display(full_balances.loc[full_balances["returned_units"].gt(0) & full_balances["cancelled_units"].eq(0),sale_cols].head(3))
display(full_lines.nlargest(3,"Quantity")[sale_cols])

,ExcelRow,CustomerID,InvoiceNo,StockCode,basket_time,original_quantity,cancelled_units,returned_units,Quantity,UnitPrice,line_amount_gbp
36375,61621,12346,541431,23166,2011-01-18 10:01:00,74215,74215,0,0,1.04,0.0
390599,540423,16446,581483,23843,2011-12-09 09:15:00,80995,80995,0,0,2.08,0.0


,ExcelRow,InvoiceNo,StockCode,Quantity,UnitPrice
34087,34089,539318,21213,24,0.55
34088,34090,539318,21213,120,0.42
157029,157031,550188,22636,8,7.65
157042,157044,550188,22636,1,0.00


,CustomerID,InvoiceNo,basket_time,StockCode,quantity,amount_gbp,source_lines
183,12348,539318,2010-12-16 19:09:00,21213,144,63.6,2
7589,12457,550188,2011-04-14 18:57:00,22636,8,61.2,1


,ExcelRow,CustomerID,InvoiceNo,StockCode,basket_time,original_quantity,cancelled_units,returned_units,Quantity,UnitPrice,line_amount_gbp
260,272,13767,536395,21314,2010-12-01 10:47:00,8,1,0,7,2.10,14.70
286,298,17924,536397,35004B,2010-12-01 10:51:00,12,5,0,7,4.65,32.55
850,886,17897,536488,22960,2010-12-01 12:31:00,8,6,0,2,4.25,8.50


,ExcelRow,CustomerID,InvoiceNo,StockCode,basket_time,original_quantity,cancelled_units,returned_units,Quantity,UnitPrice,line_amount_gbp
64,67,15100,536374,21258,2010-12-01 09:09:00,32,0,12,32,10.95,350.4
95,98,14688,536378,21212,2010-12-01 09:37:00,120,0,2,120,0.42,50.4
97,100,14688,536378,21977,2010-12-01 09:37:00,24,0,1,24,0.55,13.2


,ExcelRow,CustomerID,InvoiceNo,StockCode,basket_time,original_quantity,cancelled_units,returned_units,Quantity,UnitPrice,line_amount_gbp
303385,421634,12901,573008,84077,2011-10-27 12:26:00,4800,0,0,4800,0.21,1008.00
140979,206123,13135,554868,22197,2011-05-27 10:52:00,4300,0,0,4300,0.72,3096.00
60437,97434,18087,544612,22053,2011-02-22 10:43:00,3906,0,0,3906,0.82,3202.92


## Observe a cancellation crossing a monthly boundary

Pick a sale from the training window that a later month's credit cancels. Rebuild that line twice: at the first instant of the credit month and just after the credit timestamp. Both rows refer to the same source sale. On invoice 542375, Excel row 73737 still has its original 12 units at 2011-02-01 00:00. Credit C542781 at 09:38 (row 77743) reduces it to zero. The earlier quantity is the evidence available for a prediction at that boundary. The later quantity illustrates how the operational history changes when the credit becomes visible. If a full-workbook retrospective table were sliced after the fact, the earlier recommendation would see future information.

## 实际核查一笔跨月取消

找到训练期内一笔在次月被取消的销售。分别在贷项所在月第一天零点、贷项生效后的下一瞬回放同一个原始销售行。原单 542375 的第 73737 行在 2011-02-01 00:00 仍有原来的 12 件；到当天 09:38，贷项 C542781（第 77743 行）将其冲至零件。较早快照的件数就是当时可用于预测的证据。较晚快照的件数显示贷项可见以后业务历史如何变化。若事后从全期回溯结果直接筛选早期日期，则会把后来的信息泄漏给较早的推荐。

In [21]:
crossing = allocations.loc[
    allocations["status"].eq("cancellation") & allocations["credit_time"].lt(train_end)
    & allocations["purchase_time"].dt.to_period("M").ne(allocations["credit_time"].dt.to_period("M"))
].sort_values("credit_time").iloc[0]
boundary = crossing["credit_time"].to_period("M").start_time
asof_rows = []
for label, end in [("month_start", boundary), ("credit_included", crossing["credit_time"] + pd.Timedelta(seconds=1))]:
    balances_at_end = purchase_line_balances(ledger_b, allocations, cutoff=end)
    sample = balances_at_end.loc[balances_at_end["ExcelRow"].eq(crossing["purchase_excel_row"]),sale_cols].copy()
    sample.insert(0, "view", label)
    sample.insert(1, "exclusive_cutoff", end)
    asof_rows.append(sample)
display(crossing[["purchase_invoice_no","purchase_time","purchase_excel_row","credit_invoice_no",
                  "credit_time","credit_excel_row","credited_units"]].to_frame("value"))
boundary_trace = pd.concat(asof_rows, ignore_index=True)
display(boundary_trace)
assert len(boundary_trace) == 2 and boundary_trace["Quantity"].iloc[0] > boundary_trace["Quantity"].iloc[1]

,value
purchase_invoice_no,542375
purchase_time,2011-01-27 13:42:00
purchase_excel_row,73737
credit_invoice_no,C542781
credit_time,2011-02-01 09:38:00
credit_excel_row,77743
credited_units,12


,view,exclusive_cutoff,ExcelRow,CustomerID,InvoiceNo,StockCode,basket_time,original_quantity,cancelled_units,returned_units,Quantity,UnitPrice,line_amount_gbp
0,month_start,2011-02-01 00:00:00,73737,12626,542375,21731,2011-01-27 13:42:00,12,0,0,12,1.65,19.8
1,credit_included,2011-02-01 09:38:01,73737,12626,542375,21731,2011-01-27 13:42:00,12,12,0,0,1.65,0.0


## Five decisions for the technical report

| Issue | Treatment and measurable effect |
|---|---|
| Cancellations | Link each identifiable merchandise credit to one eligible same-customer, same-SKU sale with sufficient remaining units. Within the credit country’s window, remove only the matched quantity from that sale once the credit is visible. The full replay removes 189,315 units under 1,473 cancellation records. Keep original C amounts in signed accounting. |
| Returns | Beyond the window, retain the original purchase event, mark returned units and reserve its allocation balance. Classify 5,883 credits as operational returns. Keep unmatched and anonymous credits separate. The 1,336 non-C negative lines, all zero-price and anonymous, are inventory write-offs. The workbook supplies no physical return dates. |
| Duplicates | Use all eight original fields, preserving the first row by worksheet order. Remove 5,268 exact copies. Legitimate invoice–item lines at different prices contribute separately before basket aggregation. |
| Missing customer IDs | 135,080 raw rows lack an ID. A keeps eligible guest activity. B excludes anonymous records. A guest label is used only for display. Customer-specific matching and baskets require the original ID. |
| Anomalous quantities and prices | Remove all three A bad-debt lines, including both negative-price rows. Exclude zero quantities, nonpositive prices and documented service codes. Retain genuine paid lines, including sub-penny merchandise and remaining bulk purchases. No IQR or three-standard-deviation cap is applied. |

## 技术报告中五类数据问题的处理结论

| 问题 | 处理与实测影响 |
|---|---|
| 取消交易 | 对客户已知的商品贷项，寻找同客户、同商品且可逆余额足够的一条销售。间隔在贷项所在地阈值以内时，从贷项可见起仅扣除该原单相应件数。全期回放共 1,473 条取消、扣除 189,315 件，带符号账本仍计入原 C 金额。 |
| 退货 | 间隔超过相应阈值时保留原购买事件，记录退回件数并占用可逆额度。共 5,883 条归入操作性退货。未匹配与匿名贷项单列。1,336 条非 C 负数量行均零价、匿名，按库存核销处理。原表无法给出实物退回日期。 |
| 重复交易明细 | 以八个原始字段全部相同判断重复，按工作表顺序保留首条，剔除 5,268 条重复副本。同票同商品但不同价的有效行先分别计额，再按购物篮聚合。 |
| 缺失客户编号 | 原始缺编号 135,080 行。账本 A 保留合格的匿名活动，B 排除匿名行；“Guest/Unknown”只作显示。按客户配对与构造购买篮必须有真实编号。 |
| 异常数量或价格 | 全部排除三条 A 前缀坏账，其中含两条负价；剔除零数量、非正价格和有依据的服务编码。保留实物商品的亚便士价格及取消后仍有效的大宗购买。不按四分位距或三倍标准差机械截断。 |

## Audit the final decision for every original row

Join original sale eligibility, signed ledger membership and matched credit quantities by `ExcelRow`. A sale can have both returned and cancelled units; the returned quantity remains eligible for purchase events. Inspect `gross_eligibility_rule` for its pre-credit screen and `first_rule`, `credit_status`, `cancelled_units`, `retained_quantity` and `final_action` for its final treatment. In `cleaning_steps.csv`, `decision_rows` counts disjoint final decisions, while `raw_flagged` can overlap. The raw denominator remains 541,909 lines.

## 核对每条原始明细的最终决定

按 `ExcelRow` 连接原销售资格、两份带符号账本的归属及已匹配的冲销件数。同一销售可以同时有退货和取消；退货的部分仍计入购买事件。`gross_eligibility_rule` 解释贷项前的筛选，再用 `first_rule`、`credit_status`、`cancelled_units`、`retained_quantity` 与 `final_action` 检查最终处理。`cleaning_steps.csv` 中 `decision_rows` 是互斥的最终决定数，而 `raw_flagged` 可能重叠；原始分母仍为 541,909 行。

In [22]:
def snapshot_decisions(raw, gross_steps, gross_decisions, ledger, allocations, balances):
    """Assign every source row one retrospective disposition for the recommendation table.

    Preserve the original pre-credit eligibility decision separately. Units on a
    positive source row distinguish full/partial cancellations from operational
    returns, including the case where different credits touch the same sale.
    Ledger membership describes a separate signed-accounting view.
    """
    frame = raw.set_index("ExcelRow")
    decisions = gross_decisions.set_index("ExcelRow").rename(columns={"first_rule": "gross_eligibility_rule"})
    first = decisions["gross_eligibility_rule"].copy()
    duplicate = first.eq("exact_duplicate")
    is_credit = frame["InvoiceNo"].astype("string").str.startswith("C", na=False)
    writeoff = ~is_credit & frame["Quantity"].lt(0)
    debt = frame["InvoiceNo"].astype("string").str.startswith("A", na=False)
    first.loc[writeoff & ~duplicate] = "nonC_negative_inventory"
    first.loc[debt & ~duplicate] = "bad_debt_document"

    credit_status = allocations.set_index("credit_excel_row")["status"].reindex(first.index)
    signed = ledger.set_index("ExcelRow")["ledger_type"].reindex(first.index)
    # A C source line outside the merchandise ledger can represent e.g. a fee.
    outside = first.eq("cancelled_invoice") & credit_status.isna()
    first.loc[outside] = "credit_outside_ledger"
    first.loc[credit_status.notna()] = "credit_" + credit_status.dropna().astype("string")
    sale = balances.set_index("ExcelRow")
    first.loc[sale.index] = "retained"
    first.loc[sale.index[sale["returned_units"].gt(0)]] = "sale_with_return"
    first.loc[sale.index[sale["cancelled_units"].gt(0)]] = "sale_partially_cancelled"
    first.loc[sale.index[sale["Quantity"].eq(0)]] = "sale_fully_cancelled"
    kept_rules = ["retained", "sale_partially_cancelled", "sale_with_return"]
    retained_ids = first.index[first.isin(kept_rules)].sort_values()
    if not retained_ids.equals(sale.index[sale["Quantity"].gt(0)].sort_values()):
        raise ValueError("Source decisions and the signed-ledger purchase snapshot disagree.")

    decisions["first_rule"] = first
    decisions["in_signed_ledger_a"] = signed.notna()
    decisions["in_signed_ledger_b"] = signed.notna() & frame["CustomerID"].notna()
    decisions["credit_status"] = credit_status
    for col in ("original_quantity", "cancelled_units", "returned_units", "Quantity"):
        name = "retained_quantity" if col == "Quantity" else col
        decisions[name] = sale[col].reindex(first.index).astype("Int64")

    counts = first.value_counts()
    rows = []
    before = len(raw)

    def add(rule, action, flagged, sources, reason_en, reason_zh):
        nonlocal before
        n = int(counts.get(rule, 0))
        removed = n if action == "exclude" else 0
        rows.append({
            "step": len(rows) + 1, "rule": rule, "action": action,
            "raw_flagged": int(flagged), "decision_rows": n, "rows_before": before,
            "rows_removed": removed, "rows_after": before - removed,
            "evidence_sheets": sources, "reason_en": reason_en, "reason_zh": reason_zh,
        })
        before -= removed

    gross = gross_steps.set_index("rule")

    def add_gross(rule):
        row = gross.loc[rule]
        add(rule, "exclude", row.raw_flagged, row.evidence_sheets, row.reason_en, row.reason_zh)

    add_gross("exact_duplicate")
    add("bad_debt_document", "exclude", debt.sum(), "P_nonpositive, case_examples",
        "A-prefixed bad-debt documents do not represent a merchandise sale or credit.",
        "A 前缀的坏账调整单不计入商品销售或商品贷项。")
    add("nonC_negative_inventory", "exclude", writeoff.sum(), "Q_nonpositive, nonC_negative, note_summary",
        "Treat non-C negative units as inventory write-offs; the observed rows are anonymous and zero-priced.",
        "非 C 负数量行按库存核销处理；原表这些行均匿名且单价为零。")
    credit_reasons = {
        "cancellation": ("Deduct linked units from the original sale at its original unit price.",
                         "在原正单上按原单价扣减匹配的取消数量。"),
        "return": ("Keep the linked purchase and record the operational return in the signed ledger.",
                   "保留原购买事件，在带符号账表记录按时限判定的退货。"),
        "unmatched": ("Keep the signed credit; there is no supported purchase link.",
                      "保留带符号贷项，缺乏可核实的原单链接。"),
        "anonymous": ("Keep the anonymous credit in view A; a customer link is unavailable.",
                      "匿名贷项进入整体视图 A，无法关联客户。"),
        "unknown_region": ("Keep the credit separately because its region has no timing threshold.",
                           "地区不明的贷项单独保留，无法应用地区时限。"),
    }
    for label, (en, zh) in credit_reasons.items():
        rule = "credit_" + label
        add(rule, "exclude", credit_status.eq(label).sum(), "credit_allocations, credit_summary", en, zh)
    for rule in gross.index:
        if rule not in {"exact_duplicate", "cancelled_invoice", "retained"}:
            add_gross(rule)
        elif rule == "cancelled_invoice":
            add("credit_outside_ledger", "exclude", (is_credit & signed.isna()).sum(),
                "C_invoices, ledger_summary, merchandise_rules.csv",
                "This C line falls outside the cleaned merchandise ledger.",
                "该 C 明细未进入清洗后的商品贷项账表。")
    sources = "credit_allocations, reversal_examples"
    add("sale_fully_cancelled", "exclude", sale["Quantity"].eq(0).sum(), sources,
        "No purchased units remain after the matched cancellation credits.", "原正单的商品件数已被匹配取消贷项全部冲减。")
    add("sale_partially_cancelled", "adjust", (sale["cancelled_units"].gt(0) & sale["Quantity"].gt(0)).sum(), sources,
        "Retain the remaining units at the original sale price.", "仅保留剩余购买件数与其按原单价计算的金额。")
    add("sale_with_return", "retain", sale["returned_units"].gt(0).sum(), sources,
        "Operational returns preserve purchased units; any cancellations are accounted for separately.",
        "按业务规则认定的退货保留原购买件数；取消数量另行处理。")
    add("retained", "retain", counts.get("retained", 0), "ledger_summary",
        "Keep the original positive merchandise purchase.", "保留原始正向商品购买。")
    audit = pd.DataFrame.from_records(rows)
    if audit["decision_rows"].sum() != len(raw):
        raise ValueError("Some source rows have no final audit rule.")
    actions = audit.set_index("rule")["action"]
    decisions["final_action"] = first.map(actions)
    return audit, decisions.reset_index()

In [23]:
gross_steps, gross_decisions = steps, decisions
ledger, balances = ledger_a, full_balances
purchases, events = full_lines, full_events
reversal = reversal_tables
steps, decisions = snapshot_decisions(raw, gross_steps, gross_decisions, ledger, allocations, balances)
display(steps[["step","rule","raw_flagged","decision_rows","rows_removed","rows_after"]])
sample_ids = [61621, 61626, 540423, 540424, 157031, 157044]
display(decisions.loc[decisions["ExcelRow"].isin(sample_ids)])
assert len(decisions) == len(raw) == decisions["ExcelRow"].nunique()

,step,rule,raw_flagged,decision_rows,rows_removed,rows_after
0,1,exact_duplicate,5268,5268,5268,536641
1,2,bad_debt_document,3,3,3,536638
2,3,nonC_negative_inventory,1336,1336,1336,535302
3,4,credit_cancellation,1473,1473,1473,533829
4,5,credit_return,5883,5883,5883,527946
5,6,credit_unmatched,1147,1147,1147,526799
6,7,credit_anonymous,162,162,162,526637
7,8,credit_unknown_region,0,0,0,526637
8,9,missing_customer_id,135080,133536,133536,393101
9,10,missing_invoice_no,0,0,0,393101


,ExcelRow,gross_eligibility_rule,first_rule,in_signed_ledger_a,in_signed_ledger_b,credit_status,original_quantity,cancelled_units,returned_units,retained_quantity,final_action
61619,61621,retained,sale_fully_cancelled,True,True,NaN,74215,74215,0,0,exclude
61624,61626,cancelled_invoice,credit_cancellation,True,True,cancellation,<NA>,<NA>,<NA>,<NA>,exclude
157029,157031,retained,sale_with_return,True,True,NaN,8,0,6,8,retain
157042,157044,zero_price_same_item,zero_price_same_item,False,False,NaN,<NA>,<NA>,<NA>,<NA>,exclude
540421,540423,retained,sale_fully_cancelled,True,True,NaN,80995,80995,0,0,exclude
540422,540424,cancelled_invoice,credit_cancellation,True,True,cancellation,<NA>,<NA>,<NA>,<NA>,exclude


## Export the signed, full-period and as-of datasets

The signed ledgers and credit allocation audit answer amount and business-status questions. Use `training_purchase_lines.csv.gz` and `training_basket_events.csv.gz` for the October 1 snapshot. The generic `purchase_lines.csv.gz` and `basket_events.csv.gz` hold final full-window quantities, including credits observed in November and December. For other historical boundaries, call `purchase_snapshot` with the required exclusive cutoff. The manifest lists the row count and SHA-256 of every emitted CSV. Run the writer below to reproduce the files in the current project folder. [Dataset attribution: Chen (2015), CC BY 4.0](https://doi.org/10.24432/C5BW33)

## 导出带符号账本、全期结果与截止点快照

带符号账本与贷项分配审计用于回答金额和业务分类问题。十月一日训练快照使用 `training_purchase_lines.csv.gz` 和 `training_basket_events.csv.gz`。通用文件 `purchase_lines.csv.gz` 与 `basket_events.csv.gz` 保存全期最终数量，已考虑十一月、十二月出现的贷项。其他历史边界用 `purchase_snapshot` 传入相应排他截止点回放。清单逐项记录导出 CSV 的行数与 SHA-256；运行下方代码即在当前项目目录重建全部文件。[数据署名：Chen（2015），CC BY 4.0](https://doi.org/10.24432/C5BW33)

In [24]:
import argparse
import gzip
import hashlib
import json
import os
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory
TRAINING_CUTOFF = "2011-10-01"
BASELINE_PATH = "results/cleaning_audit.csv"

def snapshot_counts(purchases, events):
    return {
        "paid_merchandise_lines": len(purchases),
        "basket_item_events": len(events),
        "baskets": len(events[["CustomerID", "InvoiceNo"]].drop_duplicates()),
        "customers": int(purchases["CustomerID"].nunique()),
        "stock_codes": int(purchases["StockCode"].nunique()),
        "paid_units": int(purchases["Quantity"].sum()),
        "recorded_amount_gbp": float(format(purchases["line_amount_gbp"].sum(), ".15g")),
    }

frames = {
    BASELINE_PATH: baseline,
    "results/cleaning_steps.csv": steps,
    "results/merchandise_rules.csv": merchandise_rules(),
    "data/processed/row_decisions.csv.gz": decisions,
    "data/processed/purchase_lines.csv.gz": purchases,
    "data/processed/basket_events.csv.gz": events,
    "data/processed/training_purchase_lines.csv.gz": train_lines,
    "data/processed/training_basket_events.csv.gz": train_events,
    "data/processed/signed_ledger_a.csv.gz": ledger,
    "data/processed/signed_ledger_b.csv.gz": reversal["signed_ledger_b"],
    "results/credit_allocations.csv.gz": allocations,
}
for name in ["source_audit", "source_differences", "reconciliation", "region_summary",
             "match_differences", "benchmark"]:
    frames[f"results/reversal_{name}.csv"] = reversal[name]
manifest = {
    "source": {
        "dataset_doi": "10.24432/C5BW33", "workbook_sha256": WORKBOOK_SHA256,
        "sheet": "Online Retail", "raw_fields": RAW_COLUMNS, "excel_first_data_row": 2,
        "rows": len(raw), "first_timestamp": raw["InvoiceDate"].min().isoformat(),
        "last_timestamp": raw["InvoiceDate"].max().isoformat(),
    },
    "merchandise_evidence_period": {"start": "2010-12-01", "end_exclusive": TRAINING_CUTOFF},
    "counts": {
        "after_five_rules": int(baseline.iloc[-1]["rows_after"]),
        "gross_positive_lines_before_credits": len(balances),
        "removed_total": int(steps["rows_removed"].sum()),
        "fully_cancelled_sale_lines": int(balances["Quantity"].eq(0).sum()),
        "partially_cancelled_sale_lines": int((balances["cancelled_units"].gt(0) & balances["Quantity"].gt(0)).sum()),
        "cancelled_units": int(balances["cancelled_units"].sum()),
        "cancellation_amount_at_sale_prices_gbp": float(format((balances["cancelled_units"] * balances["UnitPrice"]).sum(), ".15g")),
        **snapshot_counts(purchases, events),
    },
    "training_snapshot": {
        "start_inclusive": "2010-12-01", "end_exclusive": TRAINING_CUTOFF,
        "credit_time_end_exclusive": TRAINING_CUTOFF, **snapshot_counts(train_lines, train_events),
    },
    "event_key": ["CustomerID", "InvoiceNo", "StockCode"],
    "basket_time": "Latest original positive eligible merchandise line timestamp on the customer invoice, fixed before credit adjustments.",
    "scope": "Full exports are retrospective. Training exports replay only baskets and credits strictly before 2011-10-01. For any earlier or target cutoff use purchase_snapshot on the signed ledger and canonical allocations.",
    "amounts": "Recommendation lines use surviving units at original sale prices. Signed A/B ledgers retain each original sale/credit quantity and actual price. Do not subtract matched cancellations again from ledger net amounts.",
}

In [25]:
def save_cleaning_results(project, baseline_path, frames, manifest):
    """Stage each artifact beside its destination, then publish the checksum manifest last."""
    project, baseline_path = Path(project), Path(baseline_path)
    results = project / "results"
    frames = [(baseline_path if name == BASELINE_PATH else project / name, frame)
              for name, frame in frames.items()]
    manifest_path = results / "data_manifest.json"
    if baseline_path.suffix.lower() != ".csv":
        raise ValueError("Choose a .csv file for the five-rule audit.")
    if len({path.resolve() for path, _ in frames}) != len(frames):
        raise ValueError("Choose distinct output paths for each table.")

    with ExitStack() as stack:
        workdirs = {}
        for directory in {path.parent for path, _ in frames}:
            directory.mkdir(parents=True, exist_ok=True)
            workdirs[directory] = Path(stack.enter_context(TemporaryDirectory(dir=directory)))
        outputs = {}
        staged = []
        for path, frame in frames:
            temp = workdirs[path.parent] / path.name
            if path.suffix == ".gz":
                with temp.open("wb") as stream:
                    with gzip.GzipFile(filename="", fileobj=stream, mode="wb", mtime=0, compresslevel=6) as zipped:
                        frame.to_csv(zipped, index=False, float_format="%.15g")
            else:
                frame.to_csv(temp, index=False)
            with temp.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            outputs[os.path.relpath(path, project)] = {"rows": len(frame), "sha256": digest}
            staged.append((temp, path))
        manifest_temp = workdirs[results] / manifest_path.name
        manifest_temp.write_text(json.dumps({**manifest, "outputs": outputs}, ensure_ascii=False, indent=2) + "\n",
                                 encoding="utf-8")
        for temp, path in staged:
            temp.replace(path)
        manifest_temp.replace(manifest_path)
save_cleaning_results(project, output, frames, manifest)
written_manifest = json.loads((project/"results/data_manifest.json").read_text())
display(pd.DataFrame.from_dict(written_manifest["outputs"], orient="index").rename_axis("path"))
print("Full-window counts / 全期统计", json.dumps(manifest["counts"], ensure_ascii=False))
print("As-of counts / 截止点统计", json.dumps(manifest["training_snapshot"], ensure_ascii=False))

,rows,sha256
path,,
results/cleaning_audit.csv,5,60b3431d9ece555ec2fd84bf3653d209760edd6d111565...
results/cleaning_steps.csv,44,b70e25e23df77078de2cf21618da11e0ed5ac7cd02c15b...
results/merchandise_rules.csv,38,5943b0e2502b82a4aed19e06e18b38f0c0ea90c406e2ca...
data/processed/row_decisions.csv.gz,541909,3eb6325809e6296e4a8a0daacf41b6be8ff7ebd665f4e6...
data/processed/purchase_lines.csv.gz,390168,5ba85260701d1973e4f198fb1fe2768d75a273d11a9741...
data/processed/basket_events.csv.gz,385352,6f799d70f60ab29b754bc227419ef5d45246f5423e41f3...
data/processed/training_purchase_lines.csv.gz,262067,11e1450e297650fb23354603ba1ffe7b6b06d4c25a9d6a...
data/processed/training_basket_events.csv.gz,259676,e5c9930ced435eff8ccb7b0cd6caca3ede4bf714fa082d...
data/processed/signed_ledger_a.csv.gz,531112,f4d70eb4943829a6558395dd4f4c193134732ebbf479d5...


Full-window counts / 全期统计 {"after_five_rules": 392692, "gross_positive_lines_before_credits": 391060, "removed_total": 151741, "fully_cancelled_sale_lines": 892, "partially_cancelled_sale_lines": 568, "cancelled_units": 189315, "cancellation_amount_at_sale_prices_gbp": 349595.9, "paid_merchandise_lines": 390168, "basket_item_events": 385352, "baskets": 18331, "customers": 4330, "stock_codes": 3656, "paid_units": 4952379, "recorded_amount_gbp": 8386326.743}
As-of counts / 截止点统计 {"start_inclusive": "2010-12-01", "end_exclusive": "2011-10-01", "credit_time_end_exclusive": "2011-10-01", "paid_merchandise_lines": 262067, "basket_item_events": 259676, "baskets": 13040, "customers": 3612, "stock_codes": 3499, "paid_units": 3514634, "recorded_amount_gbp": 5935091.803}
